# Hierarchical Region-Aware Sentiment Analysis
## MARBERTv2 · Leak-Free · Safe Augmentation · Mixed Precision

```
Architecture (3-stage pipeline):

  [Text Input]
       │
  ┌────▼────────────────────────────────────────────┐
  │  MARBERTv2 Encoder (shared)                     │
  │  → token hidden states [B, L, H]                │
  └────┬────────────────────────────────────────────┘
       │
  ┌────▼─────────────────┐   ┌──────────────────────┐
  │  STAGE 1             │   │  Negation-Aware       │
  │  Region Classifier   │   │  Attention Pooling    │
  │  → P(region | x)     │   │  → sent_vec [B, H]   │
  │    [B, 4]            │   └──────────┬───────────┘
  └────┬─────────────────┘              │
       │ soft probs                     │
  ┌────▼─────────────────┐              │
  │  STAGE 2             │◄─────────────┘
  │  Probabilistic       │
  │  Region Gate         │
  │  (no gold labels)    │
  │  → gated_vec [B, H]  │
  └────┬─────────────────┘
       │
  ┌────▼─────────────────┐
  │  STAGE 3             │
  │  Sentiment Head      │
  │  → logits [B, 2]     │
  └──────────────────────┘

Loss = λ_sent · CE(sentiment)
     + λ_region · CE(region)     ← auxiliary
     + λ_con · SupCon(sentiment) ← optional contrastive

Key design principle: Region probabilities are PREDICTED, never gold.
This eliminates dialect label leakage at inference time.
```

## Cell 1 — Imports & Configuration

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'python-bidi', 'arabic-reshaper', '-q'])

CompletedProcess(args=['pip', 'install', 'python-bidi', 'arabic-reshaper', '-q'], returncode=0)

In [ ]:
import os, json, random, warnings, re, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import arabic_reshaper
from bidi.algorithm import get_display

def fix_arabic(text):
    return get_display(arabic_reshaper.reshape(text))

try:
    from torch.cuda.amp import autocast, GradScaler
    AMP_AVAILABLE = True
except ImportError:
    AMP_AVAILABLE = False

DATA_DIR   = '/content/hierarchical_data'
RESULT_DIR = '/content/hsenti_results'
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


ENCODER_NAME = 'UBC-NLP/MARBERTv2'
MAX_LEN      = 125
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 2e-5
DROPOUT      = 0.7
PATIENCE     = 4
GRAD_CLIP    = 1.0
SEED         = 42

# Multi-task loss weights
LAMBDA_SENT   = 1.0
LAMBDA_REGION = 0.3
LAMBDA_CON    = 0.0
TEMPERATURE   = 0.07

GULF_TARGET = 200

# ── Layer Freezing ─────────────────────────────────────────────
# Freeze the bottom N encoder layers to reduce overfitting.
# MARBERTv2 has 12 layers total.
# Recommended: 8-10 to fix overfitting, 0 to train everything.
FREEZE_LAYERS = 6   # ← change this number

NEGATION_MARKERS = {
    'لا','لم','لن','ليس','ليست','لست',
    'ما','مش','مو','مب','موب','ماو','مهما',
    'بدون','غير','عدم','مهو','مالو',
}

# ── Ablation config — set any flag False to remove that component ──
CFG = {
    'use_negation_pooling': True,
    'use_region_gate':      True,
    'use_contrastive':      True,
    'use_region_aux_loss':  True,
    'use_mixed_precision':  True,
    'use_augmentation':     True,
    'use_backtranslation':  True,
    'use_contextual_aug':   True,
    'use_synonym_aug':      True,
    'use_typo_aug':         True,
}

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed()
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CFG['use_mixed_precision'] and AMP_AVAILABLE and torch.cuda.is_available()
print(f'Device: {DEVICE}  |  Mixed Precision: {USE_AMP}')
print('CFG:', json.dumps(CFG, indent=2))


Device: cuda  |  Mixed Precision: True
CFG: {
  "use_negation_pooling": true,
  "use_region_gate": true,
  "use_contrastive": true,
  "use_region_aux_loss": true,
  "use_mixed_precision": true,
  "use_augmentation": true,
  "use_backtranslation": true,
  "use_contextual_aug": true,
  "use_synonym_aug": true,
  "use_typo_aug": true
}


## Cell 2 — Load Data & Label Maps

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv', encoding='utf-8-sig')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv',   encoding='utf-8-sig')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv',  encoding='utf-8-sig')

with open(f'{DATA_DIR}/dialect_map.json', encoding='utf-8') as f:
    dmap = json.load(f)
with open(f'{DATA_DIR}/region_map.json',  encoding='utf-8') as f:
    rmap = json.load(f)

DIALECT_TO_ID     = dmap['dialect_to_id']
ID_TO_DIALECT     = {int(k): v for k,v in dmap['id_to_dialect'].items()}
DIALECT_TO_REGION = rmap['dialect_to_region']
REGION_TO_ID      = rmap['region_to_id']
ID_TO_REGION      = {int(v): k for k,v in REGION_TO_ID.items()}

NUM_DIALECTS  = len(DIALECT_TO_ID)
NUM_REGIONS   = len(REGION_TO_ID)
NUM_SENTIMENT = train_df['sentiment_label'].nunique()

print(f'Dialects : {NUM_DIALECTS}')
print(f'Regions  : {NUM_REGIONS}  → {list(REGION_TO_ID.keys())}')
print(f'Sentiment: {NUM_SENTIMENT} classes')
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

# Gulf sample counts
gulf_counts = train_df[train_df['region_label']==REGION_TO_ID['gulf']] \
              .groupby('dialect_label').size()
print('\nGulf train counts:')
for did, n in gulf_counts.items():
    print(f'  {ID_TO_DIALECT[did]:<12} {n}')

# ── FIX 1: Save a clean copy of train_df BEFORE augmentation ──
# The ablation loop (Cell 13) rebuilds train_ds from train_clean_df
# so every ablation config trains on the same data, eliminating
# the augmentation-contamination bug from the original.
train_clean_df = train_df.copy()


Dialects : 18
Regions  : 4  → ['levantine', 'maghrebi', 'gulf', 'nile_basin']
Sentiment: 2 classes
Train: 5,022  Val: 1,076  Test: 1,077

Gulf train counts:
  Bahrain      35
  Emirates     92
  Kuwait       92
  Oman         25
  Qaṭar        27
  Saudi        367


## Cell 3 — Safe Arabic Augmentation Pipeline

In [ ]:
## Cell 3 — Safe Augmentation Pipeline
#
# Design principles:
#   1. NEVER remove negation markers — they flip sentiment polarity
#   2. NEVER crop — may remove the negation word
#   3. NEVER randomly delete tokens — unsafe for Arabic morphology
#   4. Synonym replacement: safe when using dialect-aware embeddings
#   5. Contextual (MLM): replaces non-critical tokens using MARBERTv2
#   6. Keyboard typo: realistic noise that preserves word count/order
#   7. Back-translation: strongest semantic preservation
#   8. All augmenters check that negation words survive

if not CFG['use_augmentation']:
    print('[SKIP] Augmentation disabled.')
else:
    subprocess.run(['pip', 'install', 'nlpaug', '-q'], check=False)
    import nlpaug.augmenter.word as naw
    import nlpaug.augmenter.char as nac

    # ── Arabic keyboard layout for typo simulation ─────────────
    # Maps each Arabic char to adjacent keys on Arabic keyboard
    ARABIC_KEYBOARD = {
        'ض': ['ص','ق'], 'ص': ['ض','ث','ق'], 'ث': ['ص','ق','ف'],
        'ق': ['ض','ص','ث','ف','ع'], 'ف': ['ث','ق','غ','ع'],
        'غ': ['ف','ع','ه'], 'ع': ['ق','ف','غ','ه','خ'],
        'ه': ['غ','ع','خ','ح'], 'خ': ['ع','ه','ح','ج'],
        'ح': ['ه','خ','ج','د'], 'ج': ['خ','ح','د'],
        'د': ['ح','ج','ذ'], 'ذ': ['د','ر'],
        'ر': ['ذ','ز'], 'ز': ['ر','و'],
        'و': ['ز','ا'], 'ا': ['و','ى','ئ'],
        'ب': ['ل','ا','ت','ن'], 'ل': ['ب','ك','ا'],
        'ا': ['ل','و','ى'], 'ت': ['ب','ن','م'],
        'ن': ['ب','ت','م','ك'], 'م': ['ت','ن','ك'],
        'ك': ['ل','ن','م','ط'], 'ط': ['ك','ظ'],
        'ظ': ['ط'], 'ي': ['ى','ئ'],
        'ى': ['ا','ي','ئ'], 'ئ': ['ى','ي'],
        'ش': ['س','ي'], 'س': ['ش','ص'],
    }

    def keyboard_typo(text, typo_prob=0.08):
        """
        Simulate realistic Arabic keyboard typos.
        Replaces a character with an adjacent key character.
        Skips negation words entirely — polarity must be preserved.
        """
        words  = text.split()
        result = []
        for word in words:
            # Never corrupt negation words
            if word in NEGATION_MARKERS:
                result.append(word)
                continue
            chars = list(word)
            for i, ch in enumerate(chars):
                if random.random() < typo_prob and ch in ARABIC_KEYBOARD:
                    chars[i] = random.choice(ARABIC_KEYBOARD[ch])
            result.append(''.join(chars))
        out = ' '.join(result)
        return out if out != text else None

    # ── Contextual augmentation via MARBERTv2 fill-mask ────────
    try:
        from transformers import pipeline as hf_pipeline
        mlm_pipe = hf_pipeline(
            'fill-mask', model=ENCODER_NAME,
            device=0 if torch.cuda.is_available() else -1,
            top_k=5,
        )
        mlm_ok = True
        print('[OK] Contextual (MLM) augmenter loaded')
    except Exception as e:
        print(f'[WARN] MLM augmenter failed: {e}')
        mlm_pipe = None; mlm_ok = False

    def contextual_augment(text, mask_prob=0.15):
        """
        Replace ~mask_prob of tokens using MARBERTv2 fill-mask.
        Skips negation words — their removal would flip sentiment.
        Safer than random swap because replacements are contextually valid.
        """
        if mlm_pipe is None:
            return None
        words = text.split()
        if len(words) < 4:
            return None
        masked    = list(words)
        positions = []
        for i, w in enumerate(words):
            if w not in NEGATION_MARKERS and random.random() < mask_prob:
                masked[i] = '[MASK]'
                positions.append(i)
        if not positions:
            return None
        try:
            masked_text = ' '.join(masked)
            results     = mlm_pipe(masked_text)
            if isinstance(results[0], list):
                for pos, res_list in zip(positions, results):
                    masked[pos] = random.choice(res_list[:3])['token_str'].strip()
            else:
                masked[positions[0]] = random.choice(results[:3])['token_str'].strip()
            out = ' '.join(masked).strip()
            return out if out != text and len(out.split()) >= 2 else None
        except Exception:
            return None

    # ── Synonym replacement (nlpaug word embeddings) ───────────
    try:
        aug_syn = naw.WordEmbsAug(
            model_type='word2vec',
            action='substitute',
            aug_p=0.15,
        )
        syn_ok = True
        print('[OK] Synonym augmenter loaded (word2vec)')
    except Exception as e:
        aug_syn = None; syn_ok = False
        print(f'[WARN] Synonym augmenter not available: {e}')

    def synonym_augment(text):
        """
        Replace non-negation words with synonyms via word embeddings.
        After augmentation, verify all original negation words survived.
        """
        if aug_syn is None:
            return None
        orig_neg = {w for w in text.split() if w in NEGATION_MARKERS}
        try:
            result = aug_syn.augment(text)
            result = result[0] if isinstance(result, list) else result
            result = result.strip()
            # Verify negation words survived
            result_neg = {w for w in result.split() if w in NEGATION_MARKERS}
            if orig_neg and not orig_neg.issubset(result_neg):
                return None   # negation word was removed — reject
            return result if result != text and len(result.split()) >= 2 else None
        except Exception:
            return None

    # ── Back-translation (strongest semantic safety) ───────────
    if CFG['use_backtranslation']:
        try:
            aug_bt = naw.BackTranslationAug(
                from_model_name='Helsinki-NLP/opus-mt-ar-en',
                to_model_name='Helsinki-NLP/opus-mt-en-ar',
                device='cuda' if torch.cuda.is_available() else 'cpu',
                batch_size=8,
            )
            bt_ok = True
            print('[OK] Back-translation loaded')
        except Exception as e:
            aug_bt = None; bt_ok = False
            print(f'[WARN] Back-translation failed: {e}')
    else:
        aug_bt = None; bt_ok = False

    def backtranslate(text):
        if aug_bt is None:
            return None
        try:
            result = aug_bt.augment(text)
            result = result[0] if isinstance(result, list) else result
            return result.strip() if result and result != text else None
        except Exception:
            return None

    # ── Master safe_augment dispatcher ────────────────────────
    def safe_augment(name, text):
        """Route to correct augmenter. Always verify output is non-empty."""
        if name == 'typo'       and CFG['use_typo_aug']:
            return keyboard_typo(text)
        if name == 'contextual' and CFG['use_contextual_aug'] and mlm_ok:
            return contextual_augment(text)
        if name == 'synonym'    and CFG['use_synonym_aug'] and syn_ok:
            return synonym_augment(text)
        if name == 'backtrans'  and CFG['use_backtranslation'] and bt_ok:
            return backtranslate(text)
        return None

    # Build active augmenter list
    ACTIVE_AUGS = []
    if CFG['use_typo_aug']:                               ACTIVE_AUGS.append('typo')
    if CFG['use_contextual_aug'] and mlm_ok:              ACTIVE_AUGS.append('contextual')
    if CFG['use_synonym_aug'] and syn_ok:                 ACTIVE_AUGS.append('synonym')
    if CFG['use_backtranslation'] and bt_ok:              ACTIVE_AUGS.append('backtrans')
    print(f'\nActive augmenters: {ACTIVE_AUGS}')

    # ── Run augmentation on Gulf low-resource dialects ─────────
    gulf_rid   = REGION_TO_ID['gulf']
    gulf_train = train_df[train_df['region_label'] == gulf_rid].copy()
    all_aug    = []

    for did in sorted(gulf_train['dialect_label'].unique()):
        dname   = ID_TO_DIALECT[did]
        dial_df = gulf_train[gulf_train['dialect_label'] == did]
        texts   = dial_df['text'].astype(str).tolist()
        # Only augment if we have at least 2-word texts
        texts   = [t for t in texts if len(t.split()) >= 3]
        needed  = GULF_TARGET - len(texts)

        if needed <= 0:
            print(f'  [{dname:<10}] {len(texts):3d} — no augmentation needed')
            continue

        if not ACTIVE_AUGS:
            print(f'  [{dname:<10}] No active augmenters, skipping.')
            continue

        print(f'  [{dname:<10}] {len(texts):3d} → {GULF_TARGET} (need {needed})')
        aug_rows = []; cycle = tries = 0

        while len(aug_rows) < needed and tries < needed * 12:
            orig = random.choice(texts)
            aug_name = ACTIVE_AUGS[cycle % len(ACTIVE_AUGS)]
            cycle += 1; tries += 1
            result = safe_augment(aug_name, orig)
            if result:
                orig_row = dial_df[dial_df['text'].astype(str) == orig]
                sent_lbl = int(orig_row['sentiment_label'].iloc[0]) \
                           if len(orig_row) else int(dial_df['sentiment_label'].mode()[0])
                aug_rows.append({
                    'text':            result,
                    'dialect_label':   did,
                    'region_label':    gulf_rid,
                    'sentiment_label': sent_lbl,
                    'augmented':       True,
                    'aug_technique':   aug_name,
                })
        print(f'    Generated {len(aug_rows)} via {set(r["aug_technique"] for r in aug_rows)}')
        all_aug.extend(aug_rows)

    if all_aug:
        aug_df = pd.DataFrame(all_aug)
        train_df['augmented']     = False
        train_df['aug_technique'] = 'original'
        for col in train_df.columns:
            if col not in aug_df.columns:
                aug_df[col] = None
        train_df = pd.concat(
            [train_df, aug_df[train_df.columns]], ignore_index=True
        ).sample(frac=1, random_state=SEED).reset_index(drop=True)
        print(f'\nTotal train after augmentation: {len(train_df):,}')
    else:
        print('No augmentation rows generated.')

    print('[OK] Augmentation complete.')


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: UBC-NLP/MARBERTv2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when lo

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

...

target.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

[OK] Back-translation loaded

Active augmenters: ['typo', 'contextual', 'backtrans']
  [Bahrain   ]  34 → 200 (need 166)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


    Generated 166 via {'contextual', 'backtrans', 'typo'}
  [Emirates  ]  90 → 200 (need 110)
    Generated 110 via {'contextual', 'backtrans', 'typo'}
  [Kuwait    ]  89 → 200 (need 111)
    Generated 111 via {'contextual', 'backtrans', 'typo'}
  [Oman      ]  24 → 200 (need 176)
    Generated 176 via {'contextual', 'backtrans', 'typo'}
  [Qaṭar     ]  26 → 200 (need 174)
    Generated 174 via {'contextual', 'backtrans', 'typo'}
  [Saudi     ] 342 — no augmentation needed

Total train after augmentation: 5,759
[OK] Augmentation complete.


## Cell 4 — Preprocessing Pipeline

In [ ]:
## Cell 4 — Arabic Text Preprocessing
#
# Minimal preprocessing — MARBERTv2 is trained on raw dialectal Arabic.
# Over-normalisation (diacritic removal, letter normalisation) hurts
# dialect-specific features. We only do the bare minimum.

def preprocess_arabic(text):
    """
    Safe preprocessing for dialectal Arabic sentiment.
    Preserves: negation words, dialect markers, emojis (sentiment signal).
    Removes: URLs, repeated punctuation, leading/trailing whitespace.
    Does NOT: normalise alef/ya, remove diacritics, stem, lemmatise.
    """
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Collapse repeated punctuation (!!!!! → !)
    text = re.sub(r'([!؟،,.\'])\1+', r'\1', text)
    # Collapse excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to all splits
for df in [train_df, val_df, test_df]:
    df['text'] = df['text'].apply(preprocess_arabic)

print('[OK] Preprocessing applied.')
print('Sample:')
for t in train_df['text'].sample(3, random_state=42):
    print(f'  {t[:100]}')


[OK] Preprocessing applied.
Sample:
  اي والله حقير من زمااان اقول ذا الكلام واتمني محاسبته اخيرا احد انتبه لخبثه هذا الحقير
  حفظك الله ورعاك واسعدك ورزقك وعافاك يالغالي
  يخي شعب كحبة محد يطلع عل حكومة خرا


## Cell 5 — Dataset

In [ ]:
class ArabicSentimentDataset(Dataset):
    """
    Returns tokenised text + negation mask + labels.
    NO dialect/region gold label is passed to the model at inference.
    Region label is only used for the auxiliary training loss.
    """
    def __init__(self, df, tokenizer, max_len, neg_markers):
        self.texts     = df['text'].astype(str).tolist()
        self.sentiment = df['sentiment_label'].astype(int).tolist()
        self.region    = df['region_label'].astype(int).tolist()
        self.tok       = tokenizer
        self.max_len   = max_len
        self.neg       = neg_markers

    def __len__(self): return len(self.texts)

    def _neg_mask(self, input_ids):
        """Binary mask: 1 at positions of negation tokens."""
        mask = torch.zeros(self.max_len, dtype=torch.float)
        for i, tid in enumerate(input_ids[1:self.max_len-1], start=1):
            tok = self.tok.decode([tid], skip_special_tokens=True).strip()
            if tok in self.neg:
                mask[i] = 1.0
        return mask

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        ids = enc['input_ids'].squeeze(0)
        return {
            'input_ids':      ids,
            'attention_mask': enc['attention_mask'].squeeze(0),
            'neg_mask':       self._neg_mask(ids.tolist()),
            'sentiment':      torch.tensor(self.sentiment[i], dtype=torch.long),
            'region':         torch.tensor(self.region[i],    dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

train_ds  = ArabicSentimentDataset(train_df,  tokenizer, MAX_LEN, NEGATION_MARKERS)
val_ds    = ArabicSentimentDataset(val_df,    tokenizer, MAX_LEN, NEGATION_MARKERS)
test_ds   = ArabicSentimentDataset(test_df,   tokenizer, MAX_LEN, NEGATION_MARKERS)

# ── FIX 2: Also build a clean dataset (pre-augmentation) ───────
# Used in Cell 13 so every ablation config gets the same training
# data, making architecture comparisons valid.
train_clean_ds = ArabicSentimentDataset(train_clean_df, tokenizer, MAX_LEN, NEGATION_MARKERS)

train_ldr = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=2, pin_memory=True)
val_ldr   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_ldr  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'[OK] DataLoaders ready. Train batches: {len(train_ldr)}')


[OK] DataLoaders ready. Train batches: 179


## Cell 6 — Hierarchical Model Architecture

In [ ]:
## Cell 6 — Hierarchical Region-Aware Sentiment Model
#
# ┌──────────────────────────────────────────────────────────────┐
# │  WHY THIS HIERARCHY IS BETTER SCIENTIFICALLY                 │
# │                                                              │
# │  1. No label leakage: region is PREDICTED, not gold-fed.     │
# │     The model must learn to identify region on its own,      │
# │     just as it would in real deployment.                     │
# │                                                              │
# │  2. Region → Sentiment is linguistically motivated:          │
# │     • مو is Gulf negation; مش is Levantine/Egyptian          │
# │     • The same surface form has different sentiment weight   │
# │       depending on the dialect region.                       │
# │                                                              │
# │  3. Probabilistic gate (soft) >> hard argmax routing:        │
# │     • When region prediction is uncertain, soft weighting    │
# │       hedges across regions rather than committing to one.   │
# │     • Differentiable end-to-end — gradients flow through     │
# │       the gate during training.                              │
# │                                                              │
# │  4. Negation-aware pooling addresses CLS limitation:         │
# │     • CLS averages all positions equally via self-attention. │
# │     • Explicit upweighting at negation positions ensures     │
# │       the sentence vector captures polarity-flipping words.  │
# └──────────────────────────────────────────────────────────────┘


class NegationAwarePooling(nn.Module):
    """
    Attention pooling over all token positions with a learnable
    additive bias at negation marker positions.

    score_i = w·tanh(h_i) + α·neg_mask_i
    pool    = softmax(score) · H

    α starts at 0.5 and adapts — if it grows during training,
    negation positions are genuinely informative for this data.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn       = nn.Linear(hidden_size, 1, bias=False)
        self.neg_weight = nn.Parameter(torch.tensor(0.5))

    def forward(self, hidden, attn_mask, neg_mask):
        scores = self.attn(hidden).squeeze(-1)          # [B, L]
        scores = scores + self.neg_weight * neg_mask    # [B, L]
        scores = scores.masked_fill(attn_mask == 0, -1e9)
        w      = F.softmax(scores, dim=-1)              # [B, L]
        return torch.bmm(w.unsqueeze(1), hidden).squeeze(1)  # [B, H]


class RegionClassifierHead(nn.Module):
    """
    Stage 1: lightweight head that predicts P(region | CLS).
    Kept shallow intentionally — we don't want it to overfit
    region perfectly and underfit the shared encoder for sentiment.
    """
    def __init__(self, hidden_size, num_regions, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_regions),
        )

    def forward(self, cls_vec):
        return self.net(cls_vec)   # [B, R] logits


class ProbabilisticRegionGate(nn.Module):
    """
    Stage 2: condition the sentence vector on predicted region probs.

    Mechanism:
      region_probs = softmax(region_logits)   [B, R]
      region_ctx   = region_probs @ W_embed    [B, H]  (learned region embeddings)
      gate         = sigmoid(W_gate(region_ctx)) [B, H]
      output       = sent_vec * (1 + gate)     [B, H]  (residual amplification)

    Why residual (1 + gate) instead of plain sigmoid gate?
      Plain gate can suppress the sentence vector to near-zero.
      Residual ensures the original representation is always preserved,
      and the region signal only adds information, never destroys it.

    Why soft probs instead of argmax?
      When P(gulf)=0.6, P(levantine)=0.4, a hard decision loses
      the uncertainty. Soft weighting propagates both signals.
    """
    def __init__(self, num_regions, hidden_size, dropout):
        super().__init__()
        # Learnable embedding per region — model learns what each region means
        self.region_embed = nn.Embedding(num_regions, hidden_size)
        self.gate_proj    = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, sent_vec, region_logits):
        """
        Args:
            sent_vec:      [B, H]  pooled sentence representation
            region_logits: [B, R]  raw logits from region head
        Returns:
            gated: [B, H]
        """
        region_probs = F.softmax(region_logits, dim=-1)       # [B, R]
        # Weighted sum of region embeddings
        all_embeds   = self.region_embed.weight                # [R, H]
        region_ctx   = torch.matmul(region_probs, all_embeds) # [B, H]
        region_ctx   = self.dropout(region_ctx)
        gate         = self.gate_proj(region_ctx)             # [B, H] in (0,1)
        return sent_vec * (1 + gate)                          # [B, H] residual


class SentimentHead(nn.Module):
    """Stage 3: final sentiment classifier + projector for SupCon."""
    def __init__(self, hidden_size, num_sentiment, dropout, proj_dim=64):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_sentiment),
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, proj_dim),
        )

    def forward(self, vec):
        logits = self.classifier(vec)
        proj   = self.projector(vec)
        proj   = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
        return logits, proj


class HierarchicalSentimentModel(nn.Module):
    """
    Full 3-stage model. No gold dialect/region labels at inference.

    Forward pass:
      1. Encode → token hidden states
      2. NegationAwarePooling → sent_vec
      3. CLS → RegionHead → region_logits (Stage 1)
      4. sent_vec + region_logits → ProbGate → gated_vec (Stage 2)
      5. gated_vec → SentimentHead → sent_logits, proj (Stage 3)
    """
    def __init__(self, num_regions, num_sentiment,
                 dropout, use_neg_pool, use_region_gate):
        super().__init__()
        self.encoder        = AutoModel.from_pretrained(ENCODER_NAME)
        H                   = self.encoder.config.hidden_size

        # ── Freeze bottom N encoder layers ────────────────────────
        # Reduces overfitting on small datasets (5K samples).
        # Frozen layers: fixed pretrained features (syntax, morphology)
        # Trainable layers: adapt to dialect sentiment task
        self._freeze_encoder(FREEZE_LAYERS)

        self.use_neg_pool    = use_neg_pool
        self.use_region_gate = use_region_gate

        self.neg_pooling    = NegationAwarePooling(H)
        self.region_head    = RegionClassifierHead(H, num_regions, dropout)
        self.region_gate    = (ProbabilisticRegionGate(num_regions, H, dropout)
                               if use_region_gate else None)
        self.sentiment_head = SentimentHead(H, num_sentiment, dropout)

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Parameters: {total:,} total  |  {trainable:,} trainable')

    def _freeze_encoder(self, n_freeze):
        """Freeze bottom n_freeze layers of the encoder."""
        if n_freeze == 0:
            print('  No encoder layers frozen — training all layers.')
            return
        # Freeze embeddings always
        for p in self.encoder.embeddings.parameters():
            p.requires_grad = False
        # Freeze bottom n_freeze transformer layers
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < n_freeze:
                for p in layer.parameters():
                    p.requires_grad = False
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        print(f'  Frozen layers: {n_freeze}/12')
        print(f'  Trainable params: {trainable:,}  |  Frozen: {frozen:,}')

    def forward(self, input_ids, attention_mask, neg_mask):
        """
        Returns:
          sent_logits:   [B, num_sentiment]
          region_logits: [B, num_regions]
          proj:          [B, 64]  for contrastive loss
        """
        # ── Encode ────────────────────────────────────────────────
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask)
        hidden = out.last_hidden_state   # [B, L, H]
        cls    = hidden[:, 0]            # [B, H]

        # ── Stage 1: Region prediction from CLS ───────────────────
        region_logits = self.region_head(cls)   # [B, R]

        # ── Stage 2: Negation-aware sentence vector ───────────────
        if self.use_neg_pool:
            sent_vec = self.neg_pooling(hidden, attention_mask, neg_mask)
        else:
            sent_vec = cls   # fallback: CLS only

        # ── Stage 2: Probabilistic region gate ────────────────────
        if self.use_region_gate and self.region_gate is not None:
            # Uses PREDICTED region probs — no gold label leakage
            sent_vec = self.region_gate(sent_vec, region_logits)

        # ── Stage 3: Sentiment classification ─────────────────────
        sent_logits, proj = self.sentiment_head(sent_vec)

        return sent_logits, region_logits, proj


model = HierarchicalSentimentModel(
    num_regions=NUM_REGIONS,
    num_sentiment=NUM_SENTIMENT,
    dropout=DROPOUT,
    use_neg_pool=CFG['use_negation_pooling'],
    use_region_gate=CFG['use_region_gate'],
).to(DEVICE)
print('[OK] Model ready.')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
[OK] Model ready.


## Cell 7 — Loss Functions

In [ ]:
## Cell 7 — Loss Functions

# ── Sentiment CE: balanced data → label smoothing only ────────
sentiment_ce = nn.CrossEntropyLoss(label_smoothing=0.05)

# ── Region CE: class-weighted for region imbalance ────────────
region_counts = train_df['region_label'].values
region_present = np.unique(region_counts)
region_w = compute_class_weight('balanced', classes=region_present, y=region_counts)
region_w_full = np.zeros(NUM_REGIONS, dtype=np.float32)
for c, w in zip(region_present, region_w):
    region_w_full[int(c)] = w
region_ce = nn.CrossEntropyLoss(
    weight=torch.tensor(region_w_full).to(DEVICE),
    label_smoothing=0.1
)
print('Region class weights:', {ID_TO_REGION[i]: round(float(w), 3)
                                  for i,w in enumerate(region_w_full)})


def supcon_loss(proj, labels, temperature=TEMPERATURE):
    """
    Supervised Contrastive Loss.
    Positives = same sentiment class.
    Why include it: pulls together same-sentiment embeddings from
    different dialects — forces the encoder to build dialect-invariant
    sentiment representations.

    When to disable (CFG['use_contrastive']=False):
    - Very small batches (<16 samples)
    - When dialect variation is low and SupCon hurts calibration
    """
    N = proj.shape[0]
    if N < 2:
        return torch.tensor(0.0, device=proj.device)
    sim  = torch.matmul(proj, proj.T) / temperature   # [N, N]
    eye  = torch.eye(N, dtype=torch.bool, device=proj.device)
    sim  = sim.float().masked_fill(eye, -1e4).to(proj.dtype)
    pos  = (labels.unsqueeze(1) == labels.unsqueeze(0)) & ~eye
    if not pos.any():
        return torch.tensor(0.0, device=proj.device)
    log_d  = torch.logsumexp(sim, dim=1)
    losses = [-(sim[i][pos[i]].mean() - log_d[i])
               for i in range(N) if pos[i].any()]
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=proj.device)


# ── FIX 3: total_loss() accepts explicit flags instead of ──────
# reading from the global CFG dict at call time.
# In the original, mutating CFG mid-loop was the only way to
# switch loss terms between ablation configs — an exception mid-loop
# would leave CFG in a broken state for all subsequent calls.
# Now each ablation config passes its own flags directly.
def total_loss(sent_logits, region_logits, proj, sent_labels, region_labels,
               use_region_aux_loss=None, use_contrastive=None):
    # Default to whatever is currently set in CFG (preserves original behaviour
    # for normal training in Cell 9, where CFG is not being mutated)
    if use_region_aux_loss is None:
        use_region_aux_loss = CFG['use_region_aux_loss']
    if use_contrastive is None:
        use_contrastive = CFG['use_contrastive']

    l_sent   = LAMBDA_SENT   * sentiment_ce(sent_logits, sent_labels)
    l_region = (LAMBDA_REGION * region_ce(region_logits, region_labels)
                if use_region_aux_loss else 0.0)
    l_con    = (LAMBDA_CON    * supcon_loss(proj, sent_labels)
                if use_contrastive else 0.0)
    return l_sent + l_region + l_con

print('[OK] Loss functions ready.')


Region class weights: {'levantine': 0.791, 'maghrebi': 0.984, 'gulf': 1.047, 'nile_basin': 1.309}
[OK] Loss functions ready.


## Cell 8 — Training Loop (Mixed Precision + Early Stopping)

In [ ]:
def evaluate(model, loader):
    model.eval()
    sent_preds, sent_trues = [], []
    reg_preds,  reg_trues  = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sl, rl, _ = model(ids, mask, neg)
            sent_preds.extend(sl.argmax(-1).cpu().tolist())
            sent_trues.extend(batch['sentiment'].tolist())
            reg_preds.extend(rl.argmax(-1).cpu().tolist())
            reg_trues.extend(batch['region'].tolist())

    sent_acc = float(accuracy_score(sent_trues, sent_preds))
    sent_f1  = float(f1_score(sent_trues, sent_preds, average='macro', zero_division=0))
    reg_f1   = float(f1_score(reg_trues,  reg_preds,  average='macro', zero_division=0))
    return sent_acc, sent_f1, reg_f1


def train(model, use_region_aux_loss=None, use_contrastive=None):
    # Proper AdamW: weight decay on weights only
    decay    = [p for n,p in model.named_parameters()
                if p.requires_grad and 'bias' not in n and 'LayerNorm' not in n]
    no_decay = [p for n,p in model.named_parameters()
                if p.requires_grad and ('bias' in n or 'LayerNorm' in n)]
    opt = AdamW([
        {'params': decay,    'weight_decay': 0.01},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=LR)

    total_steps = len(train_ldr) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    # Mixed precision scaler
    scaler = GradScaler() if USE_AMP else None

    best_f1, best_state, wait = 0.0, None, 0
    history = []

    for ep in range(1, EPOCHS + 1):
        model.train()
        ep_loss = 0.0

        for batch in train_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sent = batch['sentiment'].to(DEVICE)
            reg  = batch['region'].to(DEVICE)

            opt.zero_grad()

            if USE_AMP:
                with autocast():
                    sl, rl, proj = model(ids, mask, neg)
                    loss = total_loss(sl, rl, proj, sent, reg,
                                      use_region_aux_loss=use_region_aux_loss,
                                      use_contrastive=use_contrastive)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                sl, rl, proj = model(ids, mask, neg)
                loss = total_loss(sl, rl, proj, sent, reg,
                                  use_region_aux_loss=use_region_aux_loss,
                                  use_contrastive=use_contrastive)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                opt.step()

            scheduler.step()
            ep_loss += loss.item()

        ep_loss /= len(train_ldr)
        tr_acc, tr_f1, tr_rfj = evaluate(model, train_ldr)
        va_acc, va_f1, va_rf1 = evaluate(model, val_ldr)
        gap = tr_acc - va_acc

        history.append(dict(
            epoch=ep, loss=ep_loss,
            train_acc=tr_acc, val_acc=va_acc,
            train_f1=tr_f1,   val_f1=va_f1,
            train_reg_f1=tr_rfj, val_reg_f1=va_rf1,
        ))

        neg_w = model.neg_pooling.neg_weight.item() if CFG['use_negation_pooling'] else 0
        print(f'Ep{ep:02d} | loss={ep_loss:.4f} '
              f'sent_f1={va_f1:.4f} reg_f1={va_rf1:.4f} '
              f'gap={gap:.3f} neg_α={neg_w:.3f}')

        if va_f1 > best_f1:
            best_f1    = va_f1
            best_state = {k: v.clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'Early stop ep{ep}. Best val F1={best_f1:.4f}')
                break

    if best_state:
        model.load_state_dict(best_state)
    return history

print('[OK] Training loop ready.')


[OK] Training loop ready.


## Cell 9 — Train

In [ ]:
print('='*65)
print('Hierarchical Region-Aware Sentiment Model')
print(f'  Stage 1 (Region)    : RegionClassifierHead')
print(f'  Stage 2 (Gate)      : ProbabilisticRegionGate = {CFG["use_region_gate"]}')
print(f'  Negation pooling    : {CFG["use_negation_pooling"]}')
print(f'  Contrastive loss    : {CFG["use_contrastive"]}')
print(f'  Region aux loss     : {CFG["use_region_aux_loss"]}')
print(f'  Mixed precision     : {USE_AMP}')
print('='*65)

history = train(model)

# ── Plot training curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Training Curves — Hierarchical Sentiment Model', fontsize=13, fontweight='bold')
epochs = [h['epoch']    for h in history]
colors = {'train': '#0077b6', 'val': '#e63946'}

axes[0].plot(epochs, [h['loss'] for h in history], color=colors['train'], linewidth=2)
axes[0].set_title('Loss'); axes[0].grid(alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

for ax, key, title in [
    (axes[1], 'f1',     'Sentiment Macro F1'),
    (axes[2], 'reg_f1', 'Region Macro F1 (aux)'),
]:
    axes[1 if key=='f1' else 2].plot(epochs, [h[f'train_{key}'] for h in history],
                                      color=colors['train'], label='Train', linewidth=2)
    axes[1 if key=='f1' else 2].plot(epochs, [h[f'val_{key}'] for h in history],
                                      color=colors['val'],   label='Val',   linewidth=2)
    axes[1 if key=='f1' else 2].set_title(title)
    axes[1 if key=='f1' else 2].legend()
    axes[1 if key=='f1' else 2].grid(alpha=0.3)
    axes[1 if key=='f1' else 2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/training_curves.png')


Hierarchical Region-Aware Sentiment Model
  Stage 1 (Region)    : RegionClassifierHead
  Stage 2 (Gate)      : ProbabilisticRegionGate = True
  Negation pooling    : True
  Contrastive loss    : True
  Region aux loss     : True
  Mixed precision     : True
Ep01 | loss=1.1349 sent_f1=0.7778 reg_f1=0.5373 gap=0.002 neg_α=0.500
Ep02 | loss=0.8731 sent_f1=0.9113 reg_f1=0.6955 gap=0.017 neg_α=0.499
Ep03 | loss=0.6005 sent_f1=0.9006 reg_f1=0.7432 gap=0.047 neg_α=0.499
Ep04 | loss=0.5035 sent_f1=0.9253 reg_f1=0.7326 gap=0.046 neg_α=0.499
Ep05 | loss=0.4512 sent_f1=0.9182 reg_f1=0.7592 gap=0.065 neg_α=0.499
Ep06 | loss=0.4151 sent_f1=0.9098 reg_f1=0.7681 gap=0.080 neg_α=0.499
Ep07 | loss=0.3805 sent_f1=0.9154 reg_f1=0.7664 gap=0.079 neg_α=0.499
Ep08 | loss=0.3629 sent_f1=0.9246 reg_f1=0.7669 gap=0.072 neg_α=0.499
Early stop ep8. Best val F1=0.9253
Saved → /content/hsenti_results/training_curves.png


## Cell 9.5 — Baseline Model (Plain MARBERTv2 Fine-Tuning)

In [ ]:
## Cell 9.5 — Baseline: Plain MARBERTv2 + single sentiment head
# No hierarchy, no region gate, no negation pooling, no aux loss.
# Trained on the same data, same seed, same splits as the hierarchical model.
# Used to measure the contribution of the hierarchical architecture.

class BaselineModel(nn.Module):
    """MARBERTv2 CLS → Dropout → Linear → sentiment logits."""
    def __init__(self, num_sentiment, dropout):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(ENCODER_NAME)
        H               = self.encoder.config.hidden_size
        # Apply same freezing as hierarchical model for fair comparison
        for p in self.encoder.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < FREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H, num_sentiment)
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0]
        return self.classifier(cls)


def evaluate_baseline(model, loader):
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            preds.extend(logits.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())
    acc = float(accuracy_score(trues, preds))
    f1  = float(f1_score(trues, preds, average='macro', zero_division=0))
    return acc, f1


def train_baseline(model):
    decay    = [p for n,p in model.named_parameters()
                if p.requires_grad and 'bias' not in n and 'LayerNorm' not in n]
    no_decay = [p for n,p in model.named_parameters()
                if p.requires_grad and ('bias' in n or 'LayerNorm' in n)]
    opt = AdamW([{'params': decay,    'weight_decay': 0.01},
                 {'params': no_decay, 'weight_decay': 0.0}], lr=LR)

    total_steps = len(train_ldr) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(
        opt, int(0.1 * total_steps), total_steps)
    scaler = GradScaler() if USE_AMP else None

    best_f1, best_state, wait = 0.0, None, 0
    history = []

    for ep in range(1, EPOCHS + 1):
        model.train(); ep_loss = 0.0
        for batch in train_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            sent = batch['sentiment'].to(DEVICE)
            opt.zero_grad()
            if USE_AMP:
                with autocast():
                    loss = nn.CrossEntropyLoss(label_smoothing=0.05)(model(ids, mask), sent)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                loss = nn.CrossEntropyLoss(label_smoothing=0.05)(model(ids, mask), sent)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
            scheduler.step()
            ep_loss += loss.item()

        ep_loss /= len(train_ldr)
        tr_acc, tr_f1 = evaluate_baseline(model, train_ldr)
        va_acc, va_f1 = evaluate_baseline(model, val_ldr)
        history.append(dict(epoch=ep, loss=ep_loss,
                            train_f1=tr_f1, val_f1=va_f1,
                            train_acc=tr_acc, val_acc=va_acc))
        print(f'  [BASELINE] Ep{ep:02d} loss={ep_loss:.4f} '
              f'tr_f1={tr_f1:.4f} val_f1={va_f1:.4f} gap={tr_acc-va_acc:.3f}')

        if va_f1 > best_f1:
            best_f1    = va_f1
            best_state = {k: v.clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'  [BASELINE] Early stop ep{ep}. Best={best_f1:.4f}')
                break

    model.load_state_dict(best_state)
    return history


# ── Train baseline ─────────────────────────────────────────────
print('='*65)
print('TRAINING BASELINE MODEL (plain MARBERTv2 fine-tuning)')
print('='*65)
set_seed(SEED)
baseline_model   = BaselineModel(NUM_SENTIMENT, DROPOUT).to(DEVICE)
history_baseline = train_baseline(baseline_model)

# ── Test baseline ──────────────────────────────────────────────
baseline_model.eval()
b_preds, b_trues = [], []
with torch.no_grad():
    for batch in test_ldr:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        logits = baseline_model(ids, mask)
        b_preds.extend(logits.argmax(-1).cpu().tolist())
        b_trues.extend(batch['sentiment'].tolist())

base_acc = float(accuracy_score(b_trues, b_preds))
base_f1  = float(f1_score(b_trues, b_preds, average='macro', zero_division=0))

print()
print('='*65)
print('BASELINE TEST RESULTS')
print('='*65)
print(classification_report(b_trues, b_preds,
      target_names=['Negative','Positive'], digits=4))

# ── Comparison table ───────────────────────────────────────────
h_preds, h_trues = [], []
model.eval()
with torch.no_grad():
    for batch in test_ldr:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        neg  = batch['neg_mask'].to(DEVICE)
        sl, _, _ = model(ids, mask, neg)
        h_preds.extend(sl.argmax(-1).cpu().tolist())
        h_trues.extend(batch['sentiment'].tolist())

hier_f1  = float(f1_score(h_trues, h_preds, average='macro', zero_division=0))
hier_acc = float(accuracy_score(h_trues, h_preds))

# ── Learning curves: both models ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Baseline vs Hierarchical — Learning Curves', fontsize=13, fontweight='bold')

ep_b = [h['epoch'] for h in history_baseline]
ep_h = [h['epoch'] for h in history]

for ax, key, title in [(axes[0], 'val_f1', 'Val Macro F1'),
                        (axes[1], 'train_f1', 'Train Macro F1')]:
    ax.plot(ep_b, [h[key] for h in history_baseline],
            color='#6c757d', linewidth=2,
            linestyle='--' if key=='train_f1' else '-',
            label='Baseline')
    ax.plot(ep_h, [h[key] for h in history],
            color='#e63946', linewidth=2,
            linestyle='--' if key=='train_f1' else '-',
            label='Hierarchical')
    ax.set_title(title); ax.legend()
    ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Macro F1')

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/baseline_vs_hierarchical.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'\nSaved → {RESULT_DIR}/baseline_vs_hierarchical.png')

# Save baseline model
torch.save(baseline_model.state_dict(), f'{RESULT_DIR}/baseline_model.pt')


TRAINING BASELINE MODEL (plain MARBERTv2 fine-tuning)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [BASELINE] Ep01 loss=0.6381 tr_f1=0.9031 val_f1=0.9087 gap=-0.006
  [BASELINE] Ep02 loss=0.3129 tr_f1=0.9435 val_f1=0.9108 gap=0.033
  [BASELINE] Ep03 loss=0.2610 tr_f1=0.9656 val_f1=0.9126 gap=0.053
  [BASELINE] Ep04 loss=0.2148 tr_f1=0.9829 val_f1=0.9154 gap=0.067
  [BASELINE] Ep05 loss=0.1912 tr_f1=0.9857 val_f1=0.9243 gap=0.061
  [BASELINE] Ep06 loss=0.1639 tr_f1=0.9914 val_f1=0.9108 gap=0.081
  [BASELINE] Ep07 loss=0.1586 tr_f1=0.9963 val_f1=0.9163 gap=0.080
  [BASELINE] Ep08 loss=0.1461 tr_f1=0.9972 val_f1=0.9089 gap=0.088
  [BASELINE] Ep09 loss=0.1400 tr_f1=0.9981 val_f1=0.9135 gap=0.085
  [BASELINE] Early stop ep9. Best=0.9243

BASELINE TEST RESULTS
              precision    recall  f1-score   support

    Negative     0.8918    0.9660    0.9274       529
    Positive     0.9643    0.8869    0.9240       548

    accuracy                         0.9257      1077
   macro avg     0.9280    0.9264    0.9257      1077
weighted avg     0.9287    0.9257    0.9256      1077


Save

## Cell 10 — Full Evaluation

In [ ]:
model.eval()
sent_preds, sent_trues, reg_preds, reg_trues, all_regions = [], [], [], [], []

with torch.no_grad():
    for batch in test_ldr:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        neg  = batch['neg_mask'].to(DEVICE)
        sl, rl, _ = model(ids, mask, neg)
        sent_preds.extend(sl.argmax(-1).cpu().tolist())
        sent_trues.extend(batch['sentiment'].tolist())
        reg_preds.extend(rl.argmax(-1).cpu().tolist())
        reg_trues.extend(batch['region'].tolist())
        all_regions.extend(batch['region'].tolist())

print('='*65)
print('SENTIMENT CLASSIFICATION REPORT (Test)')
print('='*65)
print(classification_report(sent_trues, sent_preds,
      target_names=['Negative','Positive'], digits=4))

print('='*65)
print('REGION CLASSIFICATION REPORT (Test — auxiliary)')
print('='*65)
region_names = [ID_TO_REGION[i] for i in range(NUM_REGIONS)]
print(classification_report(reg_trues, reg_preds,
      target_names=region_names, digits=4))

# ── Per-region sentiment breakdown ────────────────────────────
print('='*65)
print('SENTIMENT F1 BY REGION')
print('='*65)
rows = []
for rid in range(NUM_REGIONS):
    idx = [i for i,r in enumerate(all_regions) if r == rid]
    if not idx: continue
    p = [sent_preds[i] for i in idx]
    t = [sent_trues[i] for i in idx]
    rows.append({
        'Region': ID_TO_REGION[rid], 'N': len(idx),
        'Acc': round(accuracy_score(t,p), 4),
        'F1':  round(f1_score(t, p, average='macro', zero_division=0), 4),
    })
reg_df = pd.DataFrame(rows)
print(reg_df.to_string(index=False))
reg_df.to_csv(f'{RESULT_DIR}/per_region_sentiment.csv', index=False)

# ── Negation weight analysis ───────────────────────────────────
if CFG['use_negation_pooling']:
    neg_w = model.neg_pooling.neg_weight.item()
    print(f'\nLearned negation weight α = {neg_w:.4f}')
    print(f'  {"✓ Negation upweighted (> init 0.5)" if neg_w > 0.5 else "~ Near initialization — limited negation signal in this data"}')

# Save results
summary = {
    'sent_f1':  float(f1_score(sent_trues, sent_preds, average='macro', zero_division=0)),
    'sent_acc': float(accuracy_score(sent_trues, sent_preds)),
    'reg_f1':   float(f1_score(reg_trues, reg_preds, average='macro', zero_division=0)),
    'neg_weight': model.neg_pooling.neg_weight.item() if CFG['use_negation_pooling'] else None,
    'cfg': CFG,
}
with open(f'{RESULT_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n[DONE] Saved → {RESULT_DIR}/summary.json')


SENTIMENT CLASSIFICATION REPORT (Test)
              precision    recall  f1-score   support

    Negative     0.9011    0.9641    0.9315       529
    Positive     0.9628    0.8978    0.9292       548

    accuracy                         0.9304      1077
   macro avg     0.9319    0.9309    0.9303      1077
weighted avg     0.9325    0.9304    0.9303      1077

REGION CLASSIFICATION REPORT (Test — auxiliary)
              precision    recall  f1-score   support

   levantine     0.9262    0.7077    0.8023       390
    maghrebi     0.9338    0.8063    0.8654       315
        gulf     0.4896    0.8741    0.6277       135
  nile_basin     0.6880    0.7722    0.7276       237

    accuracy                         0.7716      1077
   macro avg     0.7594    0.7901    0.7558      1077
weighted avg     0.8213    0.7716    0.7824      1077

SENTIMENT F1 BY REGION
    Region   N    Acc     F1
 levantine 390 0.9154 0.9152
  maghrebi 315 0.9238 0.9237
      gulf 135 0.9704 0.9702
nile_basin 2

## Cell 11 — Final Comparison: Baseline vs Hierarchical

In [ ]:
## Cell 12 — Final Comparison Summary
# Run this after both models have been trained and evaluated.

# ── Re-evaluate both models on test set ───────────────────────
def get_test_results(mdl, use_neg=False):
    mdl.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in test_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE) if use_neg else None
            if use_neg:
                sl, _, _ = mdl(ids, mask, neg)
            else:
                sl = mdl(ids, mask)
            preds.extend(sl.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())
    acc = float(accuracy_score(trues, preds))
    f1  = float(f1_score(trues, preds, average='macro', zero_division=0))
    return acc, f1, preds, trues

hier_acc,  hier_f1,  hier_preds,  hier_trues  = get_test_results(model,          use_neg=True)
base_acc,  base_f1,  base_preds,  base_trues  = get_test_results(baseline_model, use_neg=False)

# ── Print comparison table ─────────────────────────────────────
print('='*65)
print('FINAL COMPARISON TABLE')
print(f'  Freeze layers: {FREEZE_LAYERS}/12')
print('='*65)
print(f'  {"Model":<32} {"Acc":>8}  {"Macro F1":>10}')
print(f'  {"-"*54}')
print(f'  {"Baseline (MARBERTv2 flat)":<32} {base_acc:>8.4f}  {base_f1:>10.4f}')
print(f'  {"Hierarchical (ours)":<32} {hier_acc:>8.4f}  {hier_f1:>10.4f}')
print(f'  {"-"*54}')
gain_acc = hier_acc - base_acc
gain_f1  = hier_f1  - base_f1
print(f'  {"Gain":<32} {gain_acc:>+8.4f}  {gain_f1:>+10.4f}')

# ── 4-panel comparison figure ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Baseline vs Hierarchical (Freeze={FREEZE_LAYERS}/12 layers)',
             fontsize=14, fontweight='bold')

COLORS = {
    'base_train': '#adb5bd', 'base_val': '#495057',
    'hier_train': '#74c0fc', 'hier_val': '#e63946',
}

ep_b = [h['epoch'] for h in history_baseline]
ep_h = [h['epoch'] for h in history]

# Panel 1: Loss
axes[0,0].plot(ep_b, [h['loss'] for h in history_baseline],
               color=COLORS['base_val'], linewidth=2, label='Baseline')
axes[0,0].plot(ep_h, [h['loss'] for h in history],
               color=COLORS['hier_val'], linewidth=2, label='Hierarchical')
axes[0,0].set_title('Training Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
axes[0,0].spines[['top','right']].set_visible(False)

# Panel 2: Val F1
axes[0,1].plot(ep_b, [h['val_f1'] for h in history_baseline],
               color=COLORS['base_val'], linewidth=2, label='Baseline Val')
axes[0,1].plot(ep_h, [h['val_f1'] for h in history],
               color=COLORS['hier_val'], linewidth=2, label='Hierarchical Val')
axes[0,1].plot(ep_b, [h['train_f1'] for h in history_baseline],
               color=COLORS['base_train'], linewidth=1.5, linestyle='--', label='Baseline Train')
axes[0,1].plot(ep_h, [h['train_f1'] for h in history],
               color=COLORS['hier_train'], linewidth=1.5, linestyle='--', label='Hierarchical Train')
axes[0,1].set_title('Macro F1 (Train -- / Val —)')
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)
axes[0,1].spines[['top','right']].set_visible(False)

# Panel 3: Overfitting gap
axes[1,0].plot(ep_b, [h['train_acc']-h['val_acc'] for h in history_baseline],
               color=COLORS['base_val'], linewidth=2, label='Baseline gap')
axes[1,0].plot(ep_h, [h['train_acc']-h['val_acc'] for h in history],
               color=COLORS['hier_val'], linewidth=2, label='Hierarchical gap')
axes[1,0].axhline(y=0.05, color='gray', linestyle='--', alpha=0.6, label='0.05 threshold')
axes[1,0].set_title('Overfit Gap (Train - Val Acc)')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
axes[1,0].spines[['top','right']].set_visible(False)

# Panel 4: Final bar chart
models_  = ['Baseline', 'Hierarchical']
f1s      = [base_f1, hier_f1]
bar_colors = [COLORS['base_val'], COLORS['hier_val']]
bars = axes[1,1].bar(models_, f1s, color=bar_colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, f1s):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                   f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[1,1].set_ylim(min(f1s)*0.97, max(f1s)*1.02)
axes[1,1].set_title('Test Macro F1')
axes[1,1].set_ylabel('Macro F1')
axes[1,1].grid(axis='y', alpha=0.3)
axes[1,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'\nSaved → {RESULT_DIR}/final_comparison.png')


FINAL COMPARISON TABLE
  Freeze layers: 6/12
  Model                                 Acc    Macro F1
  ------------------------------------------------------
  Baseline (MARBERTv2 flat)          0.9257      0.9257
  Hierarchical (ours)                0.9304      0.9303
  ------------------------------------------------------
  Gain                              +0.0046     +0.0047

Saved → /content/hsenti_results/final_comparison.png


## Cell 12 — Ablation Guide & Save

In [ ]:
## Cell 11 — Ablation Experiments & Save
#
# ── SUGGESTED ABLATION EXPERIMENTS ───────────────────────────
#
# Run Cell 13 (automated). It trains each config on the same
# pre-augmentation data so architecture comparisons are valid.
#
#  Config A — Baseline
#    use_negation_pooling=False, use_region_gate=False,
#    use_contrastive=False, use_region_aux_loss=False
#    → Plain MARBERTv2 + CLS + single CE
#
#  Config B — +Negation Pooling
#    use_negation_pooling=True, rest=False
#    → Tests if negation awareness helps
#
#  Config C — +Region Gate
#    use_negation_pooling=True, use_region_gate=True, rest=False
#    → Tests if hierarchical region conditioning helps
#    → KEY experiment: shows the scientific value of Stage 1→2
#
#  Config D — +Aux Loss
#    use_negation_pooling=True, use_region_gate=True,
#    use_region_aux_loss=True, use_contrastive=False
#    → Tests if supervised region signal improves encoder
#
#  Config E — Full Model
#    All True
#    → Best expected result
#
# ── EXPECTED ADVANTAGES ──────────────────────────────────────
# • +Region Gate should add ~1-2% F1 over baseline by providing
#   dialect-specific negation context (مو vs مش disambiguation)
# • +Negation Pooling should help on texts with explicit negation
#   (19.7% of negative samples contain negation markers in your data)
# • +Aux Loss regularises the encoder — reduces overfitting
#
# ── POSSIBLE WEAKNESSES ──────────────────────────────────────
# • If region classifier is weak (<70% F1), the gate receives
#   noisy signal. Monitor reg_f1 during training.
# • With only 5K samples, SupCon may not converge meaningfully.
#   Disable (use_contrastive=False) if it hurts val F1.
# • Gulf dialects (Bahrain, Oman, Qatar) will still be weakest
#   even after augmentation — report these separately.
#
# ── SUGGESTED HYPERPARAMETERS ─────────────────────────────────
# LR=2e-5, BATCH=32, EPOCHS=20 (early stopping), DROPOUT=0.2
# LAMBDA_SENT=1.0, LAMBDA_REGION=0.3, LAMBDA_CON=0.15
# If overfitting: increase DROPOUT to 0.3, reduce LR to 1e-5
# If underfitting: reduce DROPOUT to 0.1, increase LR to 3e-5

print('Ablation guide printed above.')
print()
print('Saving model...')
torch.save(model.state_dict(), f'{RESULT_DIR}/hierarchical_sentiment_model.pt')
torch.save({
    'model_state': model.state_dict(),
    'cfg': CFG,
    'num_regions': NUM_REGIONS,
    'num_sentiment': NUM_SENTIMENT,
    'summary': summary,
}, f'{RESULT_DIR}/checkpoint_full.pt')
print(f'[DONE] Model saved → {RESULT_DIR}/')


Ablation guide printed above.

Saving model...
[DONE] Model saved → /content/hsenti_results/


## Cell 13 — Ablation Study (Automated)

In [ ]:
## Cell 13 — Ablation Study
#
# ── FIX: All configs train on train_clean_ds (pre-augmentation) ──
# The original notebook reused train_ds, which was already built
# from the augmented train_df. Every ablation config — including
# 'Config A: Baseline' — was trained on augmented data, making it
# impossible to isolate architectural contributions from data effects.
#
# Here, make_train_loader() draws from train_clean_ds so every
# config trains on identical data. Augmentation's effect is a
# separate question and should be evaluated independently.

ABLATION_CONFIGS = [
    # (label,            use_neg_pool, use_region_gate, use_region_aux, use_contrastive)
    ('A: Baseline',      False,        False,            False,          False),
    ('B: +NegPool',      True,         False,            False,          False),
    ('C: +RegionGate',   True,         True,             False,          False),
    ('D: +AuxLoss',      True,         True,             True,           False),
    ('E: Full (SupCon)', True,         True,             True,           True ),
]

def make_model(use_neg_pool, use_region_gate):
    return HierarchicalSentimentModel(
        num_regions=NUM_REGIONS,
        num_sentiment=NUM_SENTIMENT,
        dropout=DROPOUT,
        use_neg_pool=use_neg_pool,
        use_region_gate=use_region_gate,
    ).to(DEVICE)

def get_test_f1(mdl):
    mdl.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in test_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sl, _, _ = mdl(ids, mask, neg)
            preds.extend(sl.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())
    return float(f1_score(trues, preds, average='macro', zero_division=0))

def make_train_loader():
    """
    Draws from train_clean_ds (pre-augmentation) with a fixed seed.
    Every ablation config sees the same training data — the only
    variable between configs is architecture + loss flags.
    """
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(train_clean_ds, BATCH_SIZE, shuffle=True,
                      drop_last=True, generator=g)

# ── Run all configs ────────────────────────────────────────────
ablation_rows, ablation_histories = [], {}

for (label, neg_pool, reg_gate, use_aux, use_con) in ABLATION_CONFIGS:
    print(f'\n{"="*65}\nCONFIG: {label}\n{"="*65}')

    set_seed(SEED)
    train_ldr = make_train_loader()

    m    = make_model(neg_pool, reg_gate)
    # Pass flags directly to train() and total_loss() — no global CFG mutation
    hist = train(m, use_region_aux_loss=use_aux, use_contrastive=use_con)
    test_f1     = get_test_f1(m)
    best_val_f1 = max(h['val_f1'] for h in hist)

    ablation_histories[label] = hist
    ablation_rows.append({
        'Config':  label,
        'NegPool': '✓' if neg_pool else '✗',
        'RegGate': '✓' if reg_gate else '✗',
        'AuxLoss': '✓' if use_aux  else '✗',
        'SupCon':  '✓' if use_con  else '✗',
        'Val F1':  round(best_val_f1, 4),
        'Test F1': round(test_f1,     4),
    })
    torch.save(m.state_dict(), f'{RESULT_DIR}/ablation_{label[:1]}.pt')
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Restore train_ldr to augmented dataset for any subsequent cells
g = torch.Generator(); g.manual_seed(SEED)
train_ldr = DataLoader(train_ds, BATCH_SIZE, shuffle=True, drop_last=True, generator=g)

# ── Ablation table ─────────────────────────────────────────────
abl_df = pd.DataFrame(ablation_rows)
baseline_f1 = abl_df.iloc[0]['Test F1']
abl_df['Delta'] = abl_df['Test F1'].apply(
    lambda x: f'+{x-baseline_f1:.4f}' if x >= baseline_f1 else f'{x-baseline_f1:.4f}')
print(f'\n{"="*75}\nABLATION TABLE\n{"="*75}')
print(abl_df[['Config','NegPool','RegGate','AuxLoss','SupCon',
              'Val F1','Test F1','Delta']].to_string(index=False))
abl_df.to_csv(f'{RESULT_DIR}/ablation_table.csv', index=False)

# ── Learning curves ────────────────────────────────────────────
COLORS = ['#6c757d','#0077b6','#2a9d8f','#e9c46a','#e63946']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Ablation Study — Learning Curves', fontsize=13, fontweight='bold')
for i, (label, *_) in enumerate(ABLATION_CONFIGS):
    hist = ablation_histories[label]
    ep   = [h['epoch'] for h in hist]
    axes[0].plot(ep, [h['loss']       for h in hist], color=COLORS[i], linewidth=2, label=label)
    axes[1].plot(ep, [h['val_f1']     for h in hist], color=COLORS[i], linewidth=2, label=label)
    axes[2].plot(ep, [h['val_reg_f1'] for h in hist], color=COLORS[i], linewidth=2, label=label)
for ax, t in [(axes[0],'Loss'),(axes[1],'Sentiment Val F1'),(axes[2],'Region Val F1 (aux)')]:
    ax.set_xlabel('Epoch'); ax.legend(fontsize=7); ax.set_title(t)
    ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/ablation_curves.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()

# ── Bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(abl_df['Config'], abl_df['Test F1'], color=COLORS, height=0.55, edgecolor='white')
for bar, val in zip(bars, abl_df['Test F1']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Test Macro F1'); ax.set_title('Ablation: Test Macro F1')
ax.set_xlim(abl_df['Test F1'].min()*0.98, abl_df['Test F1'].max()*1.015)
ax.grid(axis='x', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/ablation_bar.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('[DONE] Ablation complete.')



CONFIG: A: Baseline


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 163,648,839 total  |  43,925,319 trainable
Ep01 | loss=0.6896 sent_f1=0.8755 reg_f1=0.2628 gap=-0.016 neg_α=0.500
Ep02 | loss=0.4292 sent_f1=0.9198 reg_f1=0.2476 gap=0.011 neg_α=0.500
Ep03 | loss=0.2878 sent_f1=0.9318 reg_f1=0.2680 gap=0.020 neg_α=0.500
Ep04 | loss=0.2433 sent_f1=0.9227 reg_f1=0.2595 gap=0.050 neg_α=0.500
Ep05 | loss=0.2129 sent_f1=0.9209 reg_f1=0.2668 gap=0.061 neg_α=0.500
Ep06 | loss=0.1888 sent_f1=0.9117 reg_f1=0.2765 gap=0.076 neg_α=0.500
Ep07 | loss=0.1773 sent_f1=0.9191 reg_f1=0.2466 gap=0.073 neg_α=0.500
Early stop ep7. Best val F1=0.9318

CONFIG: B: +NegPool


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
Ep01 | loss=1.1306 sent_f1=0.5820 reg_f1=0.4722 gap=0.047 neg_α=0.500
Ep02 | loss=0.8848 sent_f1=0.9215 reg_f1=0.5450 gap=0.006 neg_α=0.499
Ep03 | loss=0.6048 sent_f1=0.9234 reg_f1=0.6903 gap=0.030 neg_α=0.499
Ep04 | loss=0.5167 sent_f1=0.9300 reg_f1=0.7645 gap=0.039 neg_α=0.499
Ep05 | loss=0.4646 sent_f1=0.8987 reg_f1=0.7783 gap=0.079 neg_α=0.499
Ep06 | loss=0.4320 sent_f1=0.9255 reg_f1=0.7795 gap=0.064 neg_α=0.499
Ep07 | loss=0.4019 sent_f1=0.9190 reg_f1=0.7763 gap=0.075 neg_α=0.499
Ep08 | loss=0.3752 sent_f1=0.9043 reg_f1=0.7748 gap=0.089 neg_α=0.499
Early stop ep8. Best val F1=0.9300

CONFIG: E: Full (SupCon)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
Ep01 | loss=1.1306 sent_f1=0.5820 reg_f1=0.4722 gap=0.047 neg_α=0.500
Ep02 | loss=0.8848 sent_f1=0.9215 reg_f1=0.5450 gap=0.006 neg_α=0.499
Ep03 | loss=0.6048 sent_f1=0.9234 reg_f1=0.6903 gap=0.030 neg_α=0.499
Ep04 | loss=0.5167 sent_f1=0.9300 reg_f1=0.7645 gap=0.039 neg_α=0.499
Ep05 | loss=0.4646 sent_f1=0.8987 reg_f1=0.7783 gap=0.079 neg_α=0.499
Ep06 | loss=0.4320 sent_f1=0.9255 reg_f1=0.7795 gap=0.064 neg_α=0.499
Ep07 | loss=0.4019 sent_f1=0.9190 reg_f1=0.7763 gap=0.075 neg_α=0.499
Ep08 | loss=0.3752 sent_f1=0.9043 reg_f1=0.7748 gap=0.089 neg_α=0.499
Early stop ep8. Best val F1=0.9300

ABLATION TABLE
          Config NegPool RegGate AuxLoss SupCon  Val F1  Test F1   Delta
     A: Baseline       ✗       ✗       ✗      ✗  0.9318   0.9173 +0.0000
     B: +NegPool       ✓       ✗       ✗      ✗  0.9290   0.9229 +0.0056
  C: +RegionGate       ✓       ✓      

# ═══════════════════════════════════════════════════════════
# EXTENDED EXPERIMENTS
# Publication-Quality Analysis · Multi-Seed · Ablation · Robustness
# ═══════════════════════════════════════════════════════════

# Multi-Seed Stability Study

Runs the **original full model** with 3 different random seeds.  
All architecture, hyperparameters, loss weights, and training logic are **unchanged**.  
Reports mean ± std of Test Macro F1 and Accuracy.

In [ ]:
## ── Multi-Seed Stability Study ─────────────────────────────────────────────
# Uses the ORIGINAL HierarchicalSentimentModel + train() + CFG unchanged.
# Only the random seed changes between runs.

MULTI_SEEDS = [42, 123, 2024]

def _run_one_seed(seed):
    """Train original model from scratch with a given seed. Returns (acc, f1)."""
    set_seed(seed)

    # Fresh augmented train loader with this seed's shuffle
    g = torch.Generator(); g.manual_seed(seed)
    _train_ldr = DataLoader(train_ds, BATCH_SIZE, shuffle=True, drop_last=True, generator=g)
    _val_ldr   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False)
    _test_ldr  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)

    m = HierarchicalSentimentModel(
        num_regions=NUM_REGIONS,
        num_sentiment=NUM_SENTIMENT,
        dropout=DROPOUT,
        use_neg_pool=CFG['use_negation_pooling'],
        use_region_gate=CFG['use_region_gate'],
    ).to(DEVICE)

    # Temporarily point the global loaders used inside train()
    # We pass a local copy of the training loop logic to avoid mutating globals.
    from torch.optim import AdamW as _AdamW
    from transformers import get_linear_schedule_with_warmup as _sched

    decay    = [p for n,p in m.named_parameters()
                if p.requires_grad and 'bias' not in n and 'LayerNorm' not in n]
    no_decay = [p for n,p in m.named_parameters()
                if p.requires_grad and ('bias' in n or 'LayerNorm' in n)]
    opt      = _AdamW([{'params': decay, 'weight_decay': 0.01},
                       {'params': no_decay, 'weight_decay': 0.0}], lr=LR)
    total_steps = len(_train_ldr) * EPOCHS
    scheduler   = _sched(opt, int(0.1 * total_steps), total_steps)
    scaler      = GradScaler() if USE_AMP else None

    best_f1, best_state, wait = 0.0, None, 0

    for ep in range(1, EPOCHS + 1):
        m.train()
        for batch in _train_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sent = batch['sentiment'].to(DEVICE)
            reg  = batch['region'].to(DEVICE)
            opt.zero_grad()
            if USE_AMP:
                with autocast():
                    sl, rl, proj = m(ids, mask, neg)
                    loss = total_loss(sl, rl, proj, sent, reg)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in m.parameters() if p.requires_grad], GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                sl, rl, proj = m(ids, mask, neg)
                loss = total_loss(sl, rl, proj, sent, reg)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in m.parameters() if p.requires_grad], GRAD_CLIP)
                opt.step()
            scheduler.step()

        _, va_f1, _ = evaluate(m, _val_ldr)
        if va_f1 > best_f1:
            best_f1    = va_f1
            best_state = {k: v.clone() for k,v in m.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'  [seed={seed}] Early stop ep{ep}.')
                break
        print(f'  [seed={seed}] ep{ep:02d} val_f1={va_f1:.4f}')

    m.load_state_dict(best_state)
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in _test_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sl, _, _ = m(ids, mask, neg)
            preds.extend(sl.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())
    acc = float(accuracy_score(trues, preds))
    f1  = float(f1_score(trues, preds, average='macro', zero_division=0))
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return acc, f1


print('=' * 65)
print('MULTI-SEED STABILITY STUDY')
print('Architecture & hyperparameters: ORIGINAL (unchanged)')
print('=' * 65)

seed_rows = []
for s in MULTI_SEEDS:
    print(f'\n── Seed {s} ─────────────────────────────────────────────')
    acc, f1 = _run_one_seed(s)
    seed_rows.append({'Seed': s, 'Test Acc': round(acc, 4), 'Test Macro F1': round(f1, 4)})
    print(f'  → Acc={acc:.4f}  F1={f1:.4f}')

seed_df = pd.DataFrame(seed_rows)
mean_f1 = seed_df['Test Macro F1'].mean()
std_f1  = seed_df['Test Macro F1'].std()
mean_acc = seed_df['Test Acc'].mean()
std_acc  = seed_df['Test Acc'].std()

summary_row = pd.DataFrame([{
    'Seed': 'Mean ± Std',
    'Test Acc':      f'{mean_acc:.4f} ± {std_acc:.4f}',
    'Test Macro F1': f'{mean_f1:.4f} ± {std_f1:.4f}',
}])
seed_display = pd.concat([
    seed_df.astype(str), summary_row
], ignore_index=True)

print('\n' + '=' * 65)
print('MULTI-SEED RESULTS')
print('=' * 65)
print(seed_display.to_string(index=False))
print(f'\n  Mean Test F1 : {mean_f1:.4f}')
print(f'  Std  Test F1 : {std_f1:.4f}  ({"stable" if std_f1 < 0.005 else "moderate variance" if std_f1 < 0.015 else "high variance"})')

seed_df.to_csv(f'{RESULT_DIR}/multi_seed_results.csv', index=False)
print(f'Saved → {RESULT_DIR}/multi_seed_results.csv')

# ── Bar chart with error bar ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([str(s) for s in seed_df['Seed']], seed_df['Test Macro F1'],
              color=['#0077b6','#2a9d8f','#e9c46a'], width=0.5, edgecolor='white')
ax.axhline(mean_f1, color='#e63946', linewidth=2, linestyle='--', label=f'Mean={mean_f1:.4f}')
ax.fill_between([-0.5, len(MULTI_SEEDS)-0.5],
                mean_f1 - std_f1, mean_f1 + std_f1,
                alpha=0.15, color='#e63946', label=f'±1 Std={std_f1:.4f}')
for bar, val in zip(bars, seed_df['Test Macro F1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Random Seed'); ax.set_ylabel('Test Macro F1')
ax.set_title('Multi-Seed Stability — Test Macro F1', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
ax.set_ylim(seed_df['Test Macro F1'].min() * 0.97, seed_df['Test Macro F1'].max() * 1.02)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/multi_seed_stability.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/multi_seed_stability.png')


MULTI-SEED STABILITY STUDY
Architecture & hyperparameters: ORIGINAL (unchanged)

── Seed 42 ─────────────────────────────────────────────


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
  [seed=42] ep01 val_f1=0.6715
  [seed=42] ep02 val_f1=0.9198
  [seed=42] ep03 val_f1=0.9181
  [seed=42] ep04 val_f1=0.9135
  [seed=42] ep05 val_f1=0.9198
  [seed=42] ep06 val_f1=0.9024
  [seed=42] ep07 val_f1=0.9236
  [seed=42] ep08 val_f1=0.9098
  [seed=42] ep09 val_f1=0.9136
  [seed=42] ep10 val_f1=0.9226
  [seed=42] Early stop ep11.
  → Acc=0.9266  F1=0.9266

── Seed 123 ─────────────────────────────────────────────


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
  [seed=123] ep01 val_f1=0.6086
  [seed=123] ep02 val_f1=0.8922
  [seed=123] ep03 val_f1=0.8996
  [seed=123] ep04 val_f1=0.9172
  [seed=123] ep05 val_f1=0.9116
  [seed=123] ep06 val_f1=0.9190
  [seed=123] ep07 val_f1=0.9182
  [seed=123] ep08 val_f1=0.9043
  [seed=123] ep09 val_f1=0.9089
  [seed=123] Early stop ep10.
  → Acc=0.9211  F1=0.9211

── Seed 2024 ─────────────────────────────────────────────


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,117,824  |  Frozen: 119,723,520
Parameters: 164,242,503 total  |  44,518,983 trainable
  [seed=2024] ep01 val_f1=0.8054
  [seed=2024] ep02 val_f1=0.9291
  [seed=2024] ep03 val_f1=0.9070
  [seed=2024] ep04 val_f1=0.9255
  [seed=2024] ep05 val_f1=0.9191
  [seed=2024] Early stop ep6.
  → Acc=0.9183  F1=0.9183

MULTI-SEED RESULTS
      Seed        Test Acc   Test Macro F1
        42          0.9266          0.9266
       123          0.9211          0.9211
      2024          0.9183          0.9183
Mean ± Std 0.9220 ± 0.0042 0.9220 ± 0.0042

  Mean Test F1 : 0.9220
  Std  Test F1 : 0.0042  (stable)
Saved → /content/hsenti_results/multi_seed_results.csv
Saved → /content/hsenti_results/multi_seed_stability.png


# Ablation Study (Extended)

Systematic ablation across **8 variants** of the hierarchical model.  
Each variant differs only in the targeted component — all other settings are identical.  
Results include **Delta vs. Original** for each test F1.

In [ ]:
## ── Extended Ablation Study ─────────────────────────────────────────────────
# 8 variants tested:
#   FullModel, w/o RegGate, w/o AuxLoss, w/o SupCon,
#   w/o NegPool, Freeze4, Freeze8, Minimal Baseline
#
# All variants use ORIGINAL hyperparameters.
# Train on train_clean_ds to isolate architecture from augmentation effects.

EXTENDED_ABLATION = [
    # (label,              neg_pool, reg_gate, use_aux, use_con, freeze_n)
    ('FullModel',          True,     True,     True,    True,    FREEZE_LAYERS),
    ('w/o RegGate',        True,     False,    True,    True,    FREEZE_LAYERS),
    ('w/o AuxLoss',        True,     True,     False,   True,    FREEZE_LAYERS),
    ('w/o SupCon',         True,     True,     True,    False,   FREEZE_LAYERS),
    ('w/o NegPool',        False,    True,     True,    True,    FREEZE_LAYERS),
    ('Freeze4',            True,     True,     True,    True,    4),
    ('Freeze8',            True,     True,     True,    True,    8),
    ('MinimalBaseline',    False,    False,    False,   False,   FREEZE_LAYERS),
]


def _make_model_freeze(use_neg_pool, use_region_gate, freeze_n):
    """Build HierarchicalSentimentModel with a specific freeze depth."""
    # Temporarily patch FREEZE_LAYERS globally, restore after
    import builtins
    _orig = globals().get('FREEZE_LAYERS', 6)
    # We rebuild by subclassing — cleaner than monkey-patching the global
    m = HierarchicalSentimentModel.__new__(HierarchicalSentimentModel)
    nn.Module.__init__(m)
    from transformers import AutoModel as _AM
    m.encoder = _AM.from_pretrained(ENCODER_NAME)
    H = m.encoder.config.hidden_size
    m.use_neg_pool    = use_neg_pool
    m.use_region_gate = use_region_gate
    m.neg_pooling     = NegationAwarePooling(H)
    m.region_head     = RegionClassifierHead(H, NUM_REGIONS, DROPOUT)
    m.region_gate     = (ProbabilisticRegionGate(NUM_REGIONS, H, DROPOUT)
                         if use_region_gate else None)
    m.sentiment_head  = SentimentHead(H, NUM_SENTIMENT, DROPOUT)
    m._freeze_encoder(freeze_n)
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  Params: {total:,} total | {trainable:,} trainable')
    return m.to(DEVICE)


def _train_ablation_variant(m, use_aux, use_con):
    """Train a model variant on the CLEAN (pre-aug) train set."""
    from torch.optim import AdamW as _AdamW
    from transformers import get_linear_schedule_with_warmup as _sched

    g = torch.Generator(); g.manual_seed(SEED)
    _tl = DataLoader(train_clean_ds, BATCH_SIZE, shuffle=True, drop_last=True, generator=g)
    _vl = DataLoader(val_ds, BATCH_SIZE, shuffle=False)

    decay    = [p for n,p in m.named_parameters()
                if p.requires_grad and 'bias' not in n and 'LayerNorm' not in n]
    no_decay = [p for n,p in m.named_parameters()
                if p.requires_grad and ('bias' in n or 'LayerNorm' in n)]
    opt = _AdamW([{'params': decay, 'weight_decay': 0.01},
                  {'params': no_decay, 'weight_decay': 0.0}], lr=LR)
    total_steps = len(_tl) * EPOCHS
    scheduler   = _sched(opt, int(0.1 * total_steps), total_steps)
    scaler      = GradScaler() if USE_AMP else None

    best_f1, best_state, wait = 0.0, None, 0
    for ep in range(1, EPOCHS + 1):
        m.train()
        for batch in _tl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sent = batch['sentiment'].to(DEVICE)
            reg  = batch['region'].to(DEVICE)
            opt.zero_grad()
            if USE_AMP:
                with autocast():
                    sl, rl, proj = m(ids, mask, neg)
                    loss = total_loss(sl, rl, proj, sent, reg,
                                      use_region_aux_loss=use_aux,
                                      use_contrastive=use_con)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in m.parameters() if p.requires_grad], GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                sl, rl, proj = m(ids, mask, neg)
                loss = total_loss(sl, rl, proj, sent, reg,
                                  use_region_aux_loss=use_aux,
                                  use_contrastive=use_con)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in m.parameters() if p.requires_grad], GRAD_CLIP)
                opt.step()
            scheduler.step()

        _, va_f1, _ = evaluate(m, _vl)
        if va_f1 > best_f1:
            best_f1    = va_f1
            best_state = {k: v.clone() for k,v in m.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'    Early stop ep{ep}. Best val_f1={best_f1:.4f}')
                break
        print(f'    ep{ep:02d} val_f1={va_f1:.4f}')
    m.load_state_dict(best_state)
    return best_f1


def _test_f1_acc(m):
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for batch in test_ldr:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            neg  = batch['neg_mask'].to(DEVICE)
            sl, _, _ = m(ids, mask, neg)
            preds.extend(sl.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())
    return (float(accuracy_score(trues, preds)),
            float(f1_score(trues, preds, average='macro', zero_division=0)))


ext_abl_rows = []
for (label, neg_pool, reg_gate, use_aux, use_con, freeze_n) in EXTENDED_ABLATION:
    print(f'\n{"="*65}')
    print(f'ABLATION VARIANT: {label}')
    print(f'  NegPool={neg_pool}  RegGate={reg_gate}  AuxLoss={use_aux}  SupCon={use_con}  Freeze={freeze_n}')
    print('=' * 65)
    set_seed(SEED)
    m = _make_model_freeze(neg_pool, reg_gate, freeze_n)
    best_val = _train_ablation_variant(m, use_aux, use_con)
    acc, f1  = _test_f1_acc(m)
    ext_abl_rows.append({
        'Variant':  label,
        'NegPool':  '✓' if neg_pool else '✗',
        'RegGate':  '✓' if reg_gate else '✗',
        'AuxLoss':  '✓' if use_aux  else '✗',
        'SupCon':   '✓' if use_con  else '✗',
        'Freeze':   freeze_n,
        'Val F1':   round(best_val, 4),
        'Test F1':  round(f1, 4),
    })
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()

ext_abl_df = pd.DataFrame(ext_abl_rows)
full_f1    = ext_abl_df.loc[ext_abl_df['Variant'] == 'FullModel', 'Test F1'].values[0]
ext_abl_df['Delta vs Original'] = ext_abl_df['Test F1'].apply(
    lambda x: f'+{x - full_f1:.4f}' if x >= full_f1 else f'{x - full_f1:.4f}')

print('\n' + '=' * 75)
print('EXTENDED ABLATION TABLE')
print('=' * 75)
print(ext_abl_df[['Variant','NegPool','RegGate','AuxLoss','SupCon',
                   'Freeze','Val F1','Test F1','Delta vs Original']].to_string(index=False))
ext_abl_df.to_csv(f'{RESULT_DIR}/extended_ablation_table.csv', index=False)
print(f'\nSaved → {RESULT_DIR}/extended_ablation_table.csv')

# ── Horizontal bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
palette = ['#e63946' if v == 'FullModel' else '#0077b6'
           for v in ext_abl_df['Variant']]
bars = ax.barh(ext_abl_df['Variant'], ext_abl_df['Test F1'],
               color=palette, height=0.6, edgecolor='white')
ax.axvline(full_f1, color='#e63946', linewidth=2, linestyle='--',
           label=f'FullModel={full_f1:.4f}')
for bar, val, delta in zip(bars, ext_abl_df['Test F1'], ext_abl_df['Delta vs Original']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f} ({delta})', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Test Macro F1')
ax.set_title('Extended Ablation Study — Test Macro F1', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(ext_abl_df['Test F1'].min() * 0.97, ext_abl_df['Test F1'].max() * 1.03)
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/extended_ablation_bar.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/extended_ablation_bar.png')



ABLATION VARIANT: FullModel
  NegPool=True  RegGate=True  AuxLoss=True  SupCon=True  Freeze=6


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 44,518,983  |  Frozen: 119,723,520
  Params: 164,242,503 total | 44,518,983 trainable
    ep01 val_f1=0.5820
    ep02 val_f1=0.9180
    ep03 val_f1=0.9273
    ep04 val_f1=0.9209
    ep05 val_f1=0.9337
    ep06 val_f1=0.9144
    ep07 val_f1=0.9283
    ep08 val_f1=0.9264
    Early stop ep9. Best val_f1=0.9337

ABLATION VARIANT: w/o RegGate
  NegPool=True  RegGate=False  AuxLoss=True  SupCon=True  Freeze=6


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 8/12
  Trainable params: 30,343,239  |  Frozen: 133,899,264
  Params: 164,242,503 total | 30,343,239 trainable
    ep01 val_f1=0.5180
    ep02 val_f1=0.9151
    ep03 val_f1=0.9215
    ep04 val_f1=0.9189
    ep05 val_f1=0.9190
    ep06 val_f1=0.9209
    Early stop ep7. Best val_f1=0.9215

ABLATION VARIANT: MinimalBaseline
  NegPool=False  RegGate=False  AuxLoss=False  SupCon=False  Freeze=6


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Frozen layers: 6/12
  Trainable params: 43,925,319  |  Frozen: 119,723,520
  Params: 163,648,839 total | 43,925,319 trainable
    ep01 val_f1=0.8755
    ep02 val_f1=0.9237
    ep03 val_f1=0.9217
    ep04 val_f1=0.9182
    ep05 val_f1=0.9197
    Early stop ep6. Best val_f1=0.9237

EXTENDED ABLATION TABLE
        Variant NegPool RegGate AuxLoss SupCon  Freeze  Val F1  Test F1 Delta vs Original
      FullModel       ✓       ✓       ✓      ✓       6  0.9337   0.9276           +0.0000
    w/o RegGate       ✓       ✗       ✓      ✓       6  0.9283   0.9322           +0.0046
    w/o AuxLoss       ✓       ✓       ✗      ✓       6  0.9254   0.9183           -0.0093
     w/o SupCon       ✓       ✓       ✓      ✗       6  0.9337   0.9276           +0.0000
    w/o NegPool       ✗       ✓       ✓      ✓       6  0.9299   0.9294           +0.0018
        Freeze4       ✓       ✓       ✓      ✓       4  0.9292   0.9266           -0.0010
        Freeze8       ✓       ✓       ✓      ✓       8  0.9215 

# Overfitting Diagnosis

Analyses the **train vs. validation gap** from the original training run.  
Plots training loss and validation F1, and prints an automatic interpretation.

In [ ]:
## ── Overfitting Diagnosis ────────────────────────────────────────────────────
# Uses `history` from the original Cell 9 training run — no retraining.

epochs_h   = [h['epoch']     for h in history]
train_f1_h = [h['train_f1']  for h in history]
val_f1_h   = [h['val_f1']    for h in history]
train_acc_h= [h['train_acc'] for h in history]
val_acc_h  = [h['val_acc']   for h in history]
loss_h     = [h['loss']      for h in history]

gaps       = [ta - va for ta, va in zip(train_acc_h, val_acc_h)]
final_gap  = gaps[-1]
max_gap    = max(gaps)
mean_gap   = float(np.mean(gaps))

# ── Automatic interpretation ───────────────────────────────────────────────
if final_gap < 0.05 and max_gap < 0.10:
    interpretation = 'Gap acceptable — model generalises well.'
elif final_gap < 0.10 or max_gap < 0.15:
    interpretation = 'Mild overfitting — consider increasing dropout or L2.'
else:
    interpretation = 'Potential overfitting — train/val gap is significant.'

print('=' * 65)
print('OVERFITTING DIAGNOSIS')
print('=' * 65)
print(f'  Final epoch train-val Acc gap : {final_gap:+.4f}')
print(f'  Maximum gap (any epoch)       : {max_gap:+.4f}')
print(f'  Mean gap across all epochs    : {mean_gap:+.4f}')
print(f'  Interpretation                : {interpretation}')
print()
print(f'  Best val F1  : {max(val_f1_h):.4f}  (ep {epochs_h[val_f1_h.index(max(val_f1_h))]})')
print(f'  Final train F1: {train_f1_h[-1]:.4f}')
print(f'  Final val F1  : {val_f1_h[-1]:.4f}')

# ── 3-panel diagnostic plot ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Overfitting Diagnosis — Original Model', fontsize=13, fontweight='bold')

# Panel 1: Training loss
axes[0].plot(epochs_h, loss_h, color='#0077b6', linewidth=2)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3); axes[0].spines[['top','right']].set_visible(False)

# Panel 2: Macro F1 train vs val
axes[1].plot(epochs_h, train_f1_h, color='#0077b6', linewidth=2, label='Train F1')
axes[1].plot(epochs_h, val_f1_h,   color='#e63946', linewidth=2, label='Val F1')
axes[1].fill_between(epochs_h, val_f1_h, train_f1_h,
                     where=[t > v for t,v in zip(train_f1_h, val_f1_h)],
                     alpha=0.15, color='#e63946', label='Gap')
axes[1].set_title('Macro F1: Train vs Val')
axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].grid(alpha=0.3); axes[1].spines[['top','right']].set_visible(False)

# Panel 3: Train-val accuracy gap
axes[2].plot(epochs_h, gaps, color='#e9c46a', linewidth=2, label='Train-Val Acc Gap')
axes[2].axhline(0.05, color='#e63946', linestyle='--', linewidth=1.5, label='0.05 threshold')
axes[2].axhline(0.10, color='#9d0208', linestyle=':', linewidth=1.5, label='0.10 threshold')
axes[2].set_title('Train − Val Accuracy Gap')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Acc Gap')
axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].spines[['top','right']].set_visible(False)

# Annotation box
axes[2].text(0.98, 0.05, interpretation,
             transform=axes[2].transAxes, ha='right', va='bottom',
             fontsize=8, color='#333333',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff3cd', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/overfitting_diagnosis.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'\nSaved → {RESULT_DIR}/overfitting_diagnosis.png')

overfit_df = pd.DataFrame({
    'Epoch':     epochs_h,
    'Loss':      loss_h,
    'Train F1':  train_f1_h,
    'Val F1':    val_f1_h,
    'Train Acc': train_acc_h,
    'Val Acc':   val_acc_h,
    'Gap':       gaps,
})
overfit_df.to_csv(f'{RESULT_DIR}/overfitting_diagnosis.csv', index=False)
print(f'Saved → {RESULT_DIR}/overfitting_diagnosis.csv')


OVERFITTING DIAGNOSIS
  Final epoch train-val Acc gap : +0.0716
  Maximum gap (any epoch)       : +0.0804
  Mean gap across all epochs    : +0.0510
  Interpretation                : Mild overfitting — consider increasing dropout or L2.

  Best val F1  : 0.9253  (ep 4)
  Final train F1: 0.9963
  Final val F1  : 0.9246

Saved → /content/hsenti_results/overfitting_diagnosis.png
Saved → /content/hsenti_results/overfitting_diagnosis.csv


# Error Analysis

Analyses **misclassified examples** from the original trained model on the test set.  
Includes negation-specific analysis for Arabic markers: لا / مو / مش / ما  
Compares negation accuracy vs. overall accuracy.

In [ ]:
## ── Error Analysis ───────────────────────────────────────────────────────────
# Uses the original `model` (already trained in Cell 9). No retraining.

NEGATION_MARKERS_ANALYSIS = {'لا', 'مو', 'مش', 'ما', 'لم', 'لن', 'ليس', 'ليست'}

model.eval()
ea_preds, ea_trues, ea_probs, ea_texts, ea_regions = [], [], [], [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_ldr):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        neg  = batch['neg_mask'].to(DEVICE)
        sl, _, _ = model(ids, mask, neg)
        probs     = F.softmax(sl, dim=-1)
        conf, pred = probs.max(dim=-1)
        ea_preds.extend(pred.cpu().tolist())
        ea_trues.extend(batch['sentiment'].tolist())
        ea_probs.extend(conf.cpu().tolist())
        ea_regions.extend(batch['region'].tolist())
        # Decode texts: map batch items back to test_df rows
        start = batch_idx * BATCH_SIZE
        end   = min(start + BATCH_SIZE, len(test_df))
        ea_texts.extend(test_df['text'].iloc[start:end].tolist())

# ── Build full error analysis dataframe ───────────────────────────────────
ea_df = pd.DataFrame({
    'text':           ea_texts[:len(ea_preds)],
    'true_label':     ['Negative' if t == 0 else 'Positive' for t in ea_trues],
    'pred_label':     ['Negative' if p == 0 else 'Positive' for p in ea_preds],
    'region':         [ID_TO_REGION.get(r, str(r)) for r in ea_regions[:len(ea_preds)]],
    'confidence':     [round(c, 4) for c in ea_probs],
    'correct':        [t == p for t, p in zip(ea_trues, ea_preds)],
})

misclassified_df = ea_df[~ea_df['correct']].reset_index(drop=True)
print('=' * 65)
print('ERROR ANALYSIS')
print('=' * 65)
print(f'  Total test samples    : {len(ea_df)}')
print(f'  Correctly classified  : {ea_df["correct"].sum()}')
print(f'  Misclassified         : {len(misclassified_df)}  '
      f'({100*len(misclassified_df)/len(ea_df):.1f}%)')

print('\n── Top 15 Misclassified Examples ─────────────────────────────')
display_cols = ['text', 'true_label', 'pred_label', 'region', 'confidence']
print(misclassified_df[display_cols].head(15).to_string(index=False, max_colwidth=60))

# ── Negation analysis ──────────────────────────────────────────────────────
def _contains_negation(text):
    words = str(text).split()
    return any(w in NEGATION_MARKERS_ANALYSIS for w in words)

ea_df['has_negation'] = ea_df['text'].apply(_contains_negation)
neg_df  = ea_df[ea_df['has_negation']]
pos_df  = ea_df[~ea_df['has_negation']]

overall_acc = ea_df['correct'].mean()
neg_acc     = neg_df['correct'].mean() if len(neg_df) > 0 else float('nan')
notneg_acc  = pos_df['correct'].mean() if len(pos_df) > 0 else float('nan')

print('\n── Negation Marker Analysis ──────────────────────────────────')
print(f'  Arabic negation markers analysed: لا / مو / مش / ما / لم / لن / ليس / ليست')
print(f'  Samples with negation   : {len(neg_df):4d}  ({100*len(neg_df)/len(ea_df):.1f}%)')
print(f'  Samples without negation: {len(pos_df):4d}  ({100*len(pos_df)/len(ea_df):.1f}%)')
print(f'\n  Overall Accuracy        : {overall_acc:.4f}')
print(f'  Accuracy on neg. texts  : {neg_acc:.4f}')
print(f'  Accuracy on non-neg.    : {notneg_acc:.4f}')
delta_neg = neg_acc - overall_acc
print(f'  Delta (neg vs overall)  : {delta_neg:+.4f}  '
      f'({"negation handled well" if delta_neg > -0.02 else "negation is challenging"})')

# ── Per-region error breakdown ─────────────────────────────────────────────
print('\n── Per-Region Error Rate ──────────────────────────────────────')
region_err_rows = []
for rname in ea_df['region'].unique():
    sub = ea_df[ea_df['region'] == rname]
    region_err_rows.append({
        'Region': rname, 'N': len(sub),
        'Acc': round(sub['correct'].mean(), 4),
        'Errors': int((~sub['correct']).sum()),
        'Error%': round(100 * (~sub['correct']).mean(), 1),
    })
reg_err_df = pd.DataFrame(region_err_rows).sort_values('Error%', ascending=False)
print(reg_err_df.to_string(index=False))

# ── Most confused pairs ────────────────────────────────────────────────────
confused = misclassified_df.groupby(['true_label','pred_label']).size().reset_index(name='count')
print('\n── Confusion Pairs ────────────────────────────────────────────')
print(confused.to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────
misclassified_df.to_csv(f'{RESULT_DIR}/error_analysis_misclassified.csv', index=False)
ea_df[ea_df['has_negation']].to_csv(f'{RESULT_DIR}/error_analysis_negation.csv', index=False)
reg_err_df.to_csv(f'{RESULT_DIR}/error_analysis_per_region.csv', index=False)
print(f'\nSaved → {RESULT_DIR}/error_analysis_*.csv')

# ── Confidence distribution: correct vs wrong ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Error Analysis — Confidence Distributions', fontsize=12, fontweight='bold')

axes[0].hist(ea_df[ea_df['correct']]['confidence'], bins=20,
             color='#2a9d8f', alpha=0.75, label='Correct', edgecolor='white')
axes[0].hist(ea_df[~ea_df['correct']]['confidence'], bins=20,
             color='#e63946', alpha=0.75, label='Misclassified', edgecolor='white')
axes[0].set_title('Prediction Confidence')
axes[0].set_xlabel('Confidence'); axes[0].legend()
axes[0].grid(alpha=0.3); axes[0].spines[['top','right']].set_visible(False)

# Per-region error bar chart
reg_err_sorted = reg_err_df.sort_values('Error%')
bar_colors = ['#e63946' if e > 30 else '#e9c46a' if e > 15 else '#2a9d8f'
              for e in reg_err_sorted['Error%']]
axes[1].barh(reg_err_sorted['Region'], reg_err_sorted['Error%'],
             color=bar_colors, edgecolor='white', height=0.6)
axes[1].set_title('Error Rate by Region (%)')
axes[1].set_xlabel('Error %')
axes[1].grid(axis='x', alpha=0.3); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/error_analysis_plots.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/error_analysis_plots.png')


ERROR ANALYSIS
  Total test samples    : 1077
  Correctly classified  : 1002
  Misclassified         : 75  (7.0%)

── Top 15 Misclassified Examples ─────────────────────────────
                                                        text true_label pred_label     region  confidence
حبيت ندير مصنع صغير لقطع الغيار هنا في باتنة شربولي لمرار...   Negative   Positive   maghrebi      0.5176
عنجد انتو شعب مابيعجبكم العجب يعني بسنه بدكم دوله مدمره ت...   Positive   Negative  levantine      0.9496
                            باقي مزالو برشة متوزعين بالعاصمة   Negative   Positive   maghrebi      0.7829
                             شوية همل وهبلان ودونا ورا الشمس   Positive   Negative  levantine      0.9874
                                لك تقبرن بنات سوريا ياشهمم❤️   Positive   Negative  levantine      0.9856
         لاتخافي نمر بعجبه كلشي سنه طيب وبوكل الاخضر واليابس   Positive   Negative  levantine      0.6293
                                      لك الله يشفي هيك حكومة   Positive   Negati

# Adversarial Robustness Study

**Inference-only** evaluation — the model is NOT retrained.  
Tests robustness against lightweight Arabic noise perturbations:
- Simulated Arabic typos (keyboard adjacency)
- Punctuation noise (random insertion)
- Character elongation (tashkeel / shadda style repetition)

In [ ]:
## ── Adversarial Robustness Study ────────────────────────────────────────────
# INFERENCE ONLY — model weights are NOT updated.
# We evaluate F1 drop on corrupted versions of the test set.

import random as _rnd

# ── Perturbation functions ────────────────────────────────────────────────
_ARABIC_KB = {
    'ض': ['ص','ق'], 'ص': ['ض','ث','ق'], 'ث': ['ص','ق','ف'],
    'ق': ['ض','ص','ث','ف','ع'], 'ف': ['ث','ق','غ','ع'],
    'غ': ['ف','ع','ه'], 'ع': ['ق','ف','غ','ه','خ'],
    'ه': ['غ','ع','خ','ح'], 'خ': ['ع','ه','ح','ج'],
    'ح': ['ه','خ','ج','د'], 'ج': ['خ','ح','د'],
    'د': ['ح','ج','ذ'], 'ذ': ['د','ر'], 'ر': ['ذ','ز'],
    'ز': ['ر','و'], 'و': ['ز','ا'], 'ا': ['و','ى'],
    'ب': ['ل','ا','ت','ن'], 'ل': ['ب','ك','ا'],
    'ت': ['ب','ن','م'], 'ن': ['ب','ت','م','ك'],
    'م': ['ت','ن','ك'], 'ك': ['ل','ن','م','ط'],
    'ط': ['ك','ظ'], 'ظ': ['ط'],
}

_PUNCT = ['،', '!', '؟', '.', ',']
_ELONGATE_CHARS = 'اويى'

def _perturb_typo(text, prob=0.10):
    """Simulate Arabic keyboard typos on ~prob fraction of characters."""
    words = text.split()
    out = []
    for w in words:
        chars = list(w)
        for i, ch in enumerate(chars):
            if _rnd.random() < prob and ch in _ARABIC_KB:
                chars[i] = _rnd.choice(_ARABIC_KB[ch])
        out.append(''.join(chars))
    return ' '.join(out)

def _perturb_punct(text, n=2):
    """Insert random Arabic punctuation at random positions."""
    words = text.split()
    for _ in range(n):
        pos = _rnd.randint(0, len(words))
        words.insert(pos, _rnd.choice(_PUNCT))
    return ' '.join(words)

def _perturb_elongate(text, prob=0.12):
    """Repeat elongatable characters (ا و ي ى) with prob."""
    out = []
    for ch in text:
        out.append(ch)
        if ch in _ELONGATE_CHARS and _rnd.random() < prob:
            out.append(ch)  # elongate once
    return ''.join(out)

# ── Evaluate under perturbation ───────────────────────────────────────────
def _eval_perturbed(perturb_fn, name):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_ldr):
            start = batch_idx * BATCH_SIZE
            end   = min(start + BATCH_SIZE, len(test_df))
            orig_texts = test_df['text'].iloc[start:end].tolist()
            noisy_texts = [perturb_fn(t) for t in orig_texts]

            # Re-tokenise the noisy texts
            enc = tokenizer(
                noisy_texts,
                max_length=MAX_LEN,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            ids  = enc['input_ids'].to(DEVICE)
            mask = enc['attention_mask'].to(DEVICE)
            # Rebuild negation mask for noisy text
            neg_masks = []
            for ids_row in ids:
                neg_mask_row = torch.zeros(MAX_LEN, dtype=torch.float)
                for i, tid in enumerate(ids_row[1:MAX_LEN-1].tolist(), start=1):
                    tok = tokenizer.decode([tid], skip_special_tokens=True).strip()
                    if tok in NEGATION_MARKERS:
                        neg_mask_row[i] = 1.0
                neg_masks.append(neg_mask_row)
            neg = torch.stack(neg_masks).to(DEVICE)

            sl, _, _ = model(ids, mask, neg)
            preds.extend(sl.argmax(-1).cpu().tolist())
            trues.extend(batch['sentiment'].tolist())

    f1  = float(f1_score(trues, preds, average='macro', zero_division=0))
    acc = float(accuracy_score(trues, preds))
    return acc, f1

# ── Original (clean) scores ────────────────────────────────────────────────
_rnd.seed(SEED)
orig_acc, orig_f1 = _eval_perturbed(lambda t: t, 'Original (clean)')

print('=' * 65)
print('ADVERSARIAL ROBUSTNESS STUDY')
print('(Inference only — model NOT retrained)')
print('=' * 65)

robustness_rows = []
for perturb_fn, name in [
    (lambda t: t,               'Clean (baseline)'),
    (_perturb_typo,             'Arabic Typos'),
    (_perturb_punct,            'Punctuation Noise'),
    (_perturb_elongate,         'Character Elongation'),
    (lambda t: _perturb_typo(_perturb_punct(t), 0.08),  'Typo + Punct (combined)'),
]:
    _rnd.seed(SEED)
    acc, f1 = _eval_perturbed(perturb_fn, name)
    drop_f1  = f1  - orig_f1
    drop_pct = 100 * drop_f1 / orig_f1 if orig_f1 > 0 else 0
    robustness_rows.append({
        'Perturbation':  name,
        'Acc':           round(acc, 4),
        'F1':            round(f1, 4),
        'F1 Drop':       round(drop_f1, 4),
        'Drop %':        f'{drop_pct:.1f}%',
    })
    print(f'  {name:<30} Acc={acc:.4f}  F1={f1:.4f}  ΔF1={drop_f1:+.4f} ({drop_pct:.1f}%)')

rob_df = pd.DataFrame(robustness_rows)
print('\n' + rob_df.to_string(index=False))
rob_df.to_csv(f'{RESULT_DIR}/robustness_study.csv', index=False)
print(f'\nSaved → {RESULT_DIR}/robustness_study.csv')

# ── Bar chart: F1 under each perturbation ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
palette = ['#2a9d8f' if r == 'Clean (baseline)' else '#0077b6'
           for r in rob_df['Perturbation']]
bars = ax.bar(rob_df['Perturbation'], rob_df['F1'],
              color=palette, width=0.55, edgecolor='white')
ax.axhline(orig_f1, color='#e63946', linewidth=2, linestyle='--', label=f'Clean F1={orig_f1:.4f}')
for bar, val, pct in zip(bars, rob_df['F1'], rob_df['Drop %']):
    label_str = f'{val:.4f}\n({pct})' if pct != '0.0%' else f'{val:.4f}'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            label_str, ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Test Macro F1')
ax.set_title('Adversarial Robustness — F1 Under Perturbations', fontsize=12, fontweight='bold')
ax.set_ylim(rob_df['F1'].min() * 0.97, rob_df['F1'].max() * 1.03)
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/robustness_study.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/robustness_study.png')


ADVERSARIAL ROBUSTNESS STUDY
(Inference only — model NOT retrained)
  Clean (baseline)               Acc=0.9304  F1=0.9303  ΔF1=+0.0000 (0.0%)
  Arabic Typos                   Acc=0.8449  F1=0.8429  ΔF1=-0.0874 (-9.4%)
  Punctuation Noise              Acc=0.9220  F1=0.9219  ΔF1=-0.0084 (-0.9%)
  Character Elongation           Acc=0.9192  F1=0.9192  ΔF1=-0.0112 (-1.2%)
  Typo + Punct (combined)        Acc=0.8347  F1=0.8321  ΔF1=-0.0983 (-10.6%)

           Perturbation    Acc     F1  F1 Drop Drop %
       Clean (baseline) 0.9304 0.9303   0.0000   0.0%
           Arabic Typos 0.8449 0.8429  -0.0874  -9.4%
      Punctuation Noise 0.9220 0.9219  -0.0084  -0.9%
   Character Elongation 0.9192 0.9192  -0.0112  -1.2%
Typo + Punct (combined) 0.8347 0.8321  -0.0983 -10.6%

Saved → /content/hsenti_results/robustness_study.csv
Saved → /content/hsenti_results/robustness_study.png


# Publication Tables & Visuals

Generates:
- Confusion matrix (publication quality)
- LaTeX-ready tables for all major result tables
- CSV exports of all final results
- Summary figure combining key metrics

In [ ]:
## ── Publication Tables & Visuals ────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import itertools

# ── Re-collect test predictions from original model ───────────────────────
model.eval()
pub_preds, pub_trues = [], []
with torch.no_grad():
    for batch in test_ldr:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        neg  = batch['neg_mask'].to(DEVICE)
        sl, _, _ = model(ids, mask, neg)
        pub_preds.extend(sl.argmax(-1).cpu().tolist())
        pub_trues.extend(batch['sentiment'].tolist())

pub_acc = float(accuracy_score(pub_trues, pub_preds))
pub_f1  = float(f1_score(pub_trues, pub_preds, average='macro', zero_division=0))
CLASS_NAMES = ['Negative', 'Positive']

# ── 1. Confusion Matrix ────────────────────────────────────────────────────
cm = confusion_matrix(pub_trues, pub_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_pct, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_yticklabels(CLASS_NAMES, fontsize=12)
thresh = cm_pct.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, f'{cm[i,j]}\n({cm_pct[i,j]:.1%})',
            ha='center', va='center', fontsize=12, fontweight='bold',
            color='white' if cm_pct[i,j] > thresh else 'black')
ax.set_ylabel('True Label', fontsize=12); ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title(f'Confusion Matrix — Hierarchical Model\nAcc={pub_acc:.4f}  Macro F1={pub_f1:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/confusion_matrix.png')

# ── 2. Summary Results Table ───────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score
pub_prec = float(precision_score(pub_trues, pub_preds, average='macro', zero_division=0))
pub_rec  = float(recall_score(pub_trues, pub_preds, average='macro', zero_division=0))

summary_results = pd.DataFrame([
    {'Metric': 'Accuracy',       'Score': round(pub_acc,  4)},
    {'Metric': 'Macro F1',       'Score': round(pub_f1,   4)},
    {'Metric': 'Macro Precision','Score': round(pub_prec, 4)},
    {'Metric': 'Macro Recall',   'Score': round(pub_rec,  4)},
])

print('=' * 55)
print('FINAL TEST RESULTS — HIERARCHICAL MODEL')
print('=' * 55)
print(summary_results.to_string(index=False))
summary_results.to_csv(f'{RESULT_DIR}/final_results_summary.csv', index=False)
print(f'Saved → {RESULT_DIR}/final_results_summary.csv')

# ── 3. LaTeX Table — Final Results ────────────────────────────────────────
latex_final = summary_results.to_latex(index=False, float_format='%.4f',
    caption='Hierarchical Region-Aware Sentiment Analysis — Test Set Results.',
    label='tab:final_results', column_format='lc')
print('\n── LaTeX: Final Results Table ─────────────────────────────────')
print(latex_final)
with open(f'{RESULT_DIR}/latex_final_results.tex', 'w') as f:
    f.write(latex_final)

# ── 4. LaTeX Table — Ablation ─────────────────────────────────────────────
if 'ext_abl_df' in dir():
    abl_latex_df = ext_abl_df[['Variant','NegPool','RegGate','AuxLoss','SupCon',
                                'Val F1','Test F1','Delta vs Original']].copy()
    latex_ablation = abl_latex_df.to_latex(
        index=False,
        caption='Ablation study results. Delta computed relative to the FullModel variant.',
        label='tab:ablation',
        column_format='lccccrrr')
    print('\n── LaTeX: Ablation Table ──────────────────────────────────────')
    print(latex_ablation)
    with open(f'{RESULT_DIR}/latex_ablation.tex', 'w') as f:
        f.write(latex_ablation)

# ── 5. LaTeX Table — Multi-Seed ───────────────────────────────────────────
if 'seed_df' in dir():
    latex_seed = seed_df.to_latex(
        index=False,
        caption=f'Multi-seed stability results. Mean F1={mean_f1:.4f}, Std={std_f1:.4f}.',
        label='tab:multiseed',
        column_format='crr')
    print('\n── LaTeX: Multi-Seed Table ────────────────────────────────────')
    print(latex_seed)
    with open(f'{RESULT_DIR}/latex_multi_seed.tex', 'w') as f:
        f.write(latex_seed)

# ── 6. Publication-quality summary figure ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Hierarchical Region-Aware Sentiment Analysis — Summary',
             fontsize=13, fontweight='bold', y=1.01)

# Panel 1: Metric bar chart
metrics = summary_results['Metric'].tolist()
scores  = summary_results['Score'].tolist()
bar_colors = ['#0077b6','#e63946','#2a9d8f','#e9c46a']
bars = axes[0].bar(metrics, scores, color=bar_colors, width=0.55, edgecolor='white')
for bar, val in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(min(scores) * 0.97, 1.0)
axes[0].set_title('Test Set Metrics'); axes[0].set_ylabel('Score')
axes[0].grid(axis='y', alpha=0.3); axes[0].spines[['top','right']].set_visible(False)
axes[0].tick_params(axis='x', rotation=15)

# Panel 2: Val F1 training curve (original model)
axes[1].plot(epochs_h if 'epochs_h' in dir() else [h['epoch'] for h in history],
             val_f1_h if 'val_f1_h' in dir() else [h['val_f1'] for h in history],
             color='#e63946', linewidth=2.5, label='Val F1')
axes[1].plot(epochs_h if 'epochs_h' in dir() else [h['epoch'] for h in history],
             train_f1_h if 'train_f1_h' in dir() else [h['train_f1'] for h in history],
             color='#0077b6', linewidth=2, linestyle='--', label='Train F1')
axes[1].set_title('Training Curves (Original)'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1'); axes[1].legend()
axes[1].grid(alpha=0.3); axes[1].spines[['top','right']].set_visible(False)

# Panel 3: Confusion matrix (compact)
im2 = axes[2].imshow(cm_pct, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
axes[2].set_xticks(range(len(CLASS_NAMES))); axes[2].set_yticks(range(len(CLASS_NAMES)))
axes[2].set_xticklabels(CLASS_NAMES); axes[2].set_yticklabels(CLASS_NAMES)
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    axes[2].text(j, i, f'{cm_pct[i,j]:.1%}',
                 ha='center', va='center', fontsize=14, fontweight='bold',
                 color='white' if cm_pct[i,j] > 0.5 else 'black')
axes[2].set_title('Confusion Matrix'); axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/publication_summary_figure.png', dpi=200, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved → {RESULT_DIR}/publication_summary_figure.png')

# ── 7. Master CSV export ──────────────────────────────────────────────────
print('\n── All CSV Exports ────────────────────────────────────────────')
csv_files = [
    'final_results_summary.csv',
    'multi_seed_results.csv',
    'extended_ablation_table.csv',
    'overfitting_diagnosis.csv',
    'error_analysis_misclassified.csv',
    'error_analysis_negation.csv',
    'error_analysis_per_region.csv',
    'robustness_study.csv',
    'per_region_sentiment.csv',
]
for cf in csv_files:
    path = f'{RESULT_DIR}/{cf}'
    import os as _os
    if _os.path.exists(path):
        print(f'  ✓ {path}')
    else:
        print(f'  ✗ {path}  (not yet generated)')

print('\n── All LaTeX Exports ──────────────────────────────────────────')
for lf in ['latex_final_results.tex', 'latex_ablation.tex', 'latex_multi_seed.tex']:
    path = f'{RESULT_DIR}/{lf}'
    if _os.path.exists(path):
        print(f'  ✓ {path}')
    else:
        print(f'  ✗ {path}  (not yet generated)')


Saved → /content/hsenti_results/confusion_matrix.png
FINAL TEST RESULTS — HIERARCHICAL MODEL
         Metric  Score
       Accuracy 0.9304
       Macro F1 0.9303
Macro Precision 0.9319
   Macro Recall 0.9309
Saved → /content/hsenti_results/final_results_summary.csv

── LaTeX: Final Results Table ─────────────────────────────────
\begin{table}
\caption{Hierarchical Region-Aware Sentiment Analysis — Test Set Results.}
\label{tab:final_results}
\begin{tabular}{lc}
\toprule
Metric & Score \\
\midrule
Accuracy & 0.9304 \\
Macro F1 & 0.9303 \\
Macro Precision & 0.9319 \\
Macro Recall & 0.9309 \\
\bottomrule
\end{tabular}
\end{table}


── LaTeX: Ablation Table ──────────────────────────────────────
\begin{table}
\caption{Ablation study results. Delta computed relative to the FullModel variant.}
\label{tab:ablation}
\begin{tabular}{lccccrrr}
\toprule
Variant & NegPool & RegGate & AuxLoss & SupCon & Val F1 & Test F1 & Delta vs Original \\
\midrule
FullModel & ✓ & ✓ & ✓ & ✓ & 0.933700 & 0.927600

# Final Summary

Compact research-ready outputs from all extended experiments.

In [ ]:
## ── Final Research Summary ───────────────────────────────────────────────────

print('\n' + '═' * 75)
print('HIERARCHICAL REGION-AWARE SENTIMENT ANALYSIS — FINAL SUMMARY')
print('═' * 75)

# ── 1. Core test results ──────────────────────────────────────────────────
print('\n[1] CORE TEST RESULTS (Original Model)')
print(f'    Accuracy  : {pub_acc:.4f}')
print(f'    Macro F1  : {pub_f1:.4f}')
print(f'    Precision : {pub_prec:.4f}')
print(f'    Recall    : {pub_rec:.4f}')

# ── 2. Multi-seed stability ────────────────────────────────────────────────
if 'mean_f1' in dir() and 'std_f1' in dir():
    print(f'\n[2] MULTI-SEED STABILITY  (seeds: {MULTI_SEEDS})')
    print(f'    Mean F1 : {mean_f1:.4f}')
    print(f'    Std F1  : {std_f1:.4f}')
    print(f'    Status  : {"STABLE" if std_f1 < 0.005 else "MODERATE" if std_f1 < 0.015 else "HIGH VARIANCE"}')

# ── 3. Ablation best vs worst ──────────────────────────────────────────────
if 'ext_abl_df' in dir():
    best_var  = ext_abl_df.loc[ext_abl_df['Test F1'].idxmax(), 'Variant']
    worst_var = ext_abl_df.loc[ext_abl_df['Test F1'].idxmin(), 'Variant']
    best_f1_v = ext_abl_df['Test F1'].max()
    worst_f1_v= ext_abl_df['Test F1'].min()
    print(f'\n[3] ABLATION SUMMARY (8 variants)')
    print(f'    Best variant  : {best_var:<25} F1={best_f1_v:.4f}')
    print(f'    Worst variant : {worst_var:<25} F1={worst_f1_v:.4f}')
    print(f'    Full ablation table → {RESULT_DIR}/extended_ablation_table.csv')

# ── 4. Overfitting ────────────────────────────────────────────────────────
if 'final_gap' in dir():
    print(f'\n[4] OVERFITTING DIAGNOSIS')
    print(f'    Final train-val acc gap : {final_gap:+.4f}')
    print(f'    Interpretation          : {interpretation}')

# ── 5. Error analysis highlights ──────────────────────────────────────────
if 'overall_acc' in dir():
    print(f'\n[5] ERROR ANALYSIS')
    print(f'    Overall accuracy        : {overall_acc:.4f}')
    print(f'    Negation text accuracy  : {neg_acc:.4f}')
    print(f'    Delta (neg vs overall)  : {neg_acc - overall_acc:+.4f}')
    print(f'    Negation texts in test  : {len(neg_df)} / {len(ea_df)} ({100*len(neg_df)/len(ea_df):.1f}%)')

# ── 6. Robustness ─────────────────────────────────────────────────────────
if 'rob_df' in dir():
    worst_rob = rob_df[rob_df['Perturbation'] != 'Clean (baseline)']
    worst_rob = worst_rob.loc[worst_rob['F1'].idxmin()]
    print(f'\n[6] ROBUSTNESS (inference-only perturbations)')
    print(f'    Clean F1                 : {orig_f1:.4f}')
    print(f'    Worst perturbation       : {worst_rob["Perturbation"]}')
    print(f'    Worst F1                 : {worst_rob["F1"]:.4f}  ({worst_rob["Drop %"]} drop)')

# ── 7. Output files ────────────────────────────────────────────────────────
print(f'\n[7] ALL OUTPUTS SAVED TO: {RESULT_DIR}/')
print(f'    Figures : confusion_matrix.png, publication_summary_figure.png,')
print(f'              multi_seed_stability.png, extended_ablation_bar.png,')
print(f'              overfitting_diagnosis.png, robustness_study.png,')
print(f'              error_analysis_plots.png')
print(f'    Tables  : latex_final_results.tex, latex_ablation.tex, latex_multi_seed.tex')
print(f'    CSVs    : final_results_summary.csv, multi_seed_results.csv,')
print(f'              extended_ablation_table.csv, robustness_study.csv,')
print(f'              error_analysis_*.csv, overfitting_diagnosis.csv')
print('\n' + '═' * 75)
print('[DONE] All extended experiments complete.')
print('═' * 75)



═══════════════════════════════════════════════════════════════════════════
HIERARCHICAL REGION-AWARE SENTIMENT ANALYSIS — FINAL SUMMARY
═══════════════════════════════════════════════════════════════════════════

[1] CORE TEST RESULTS (Original Model)
    Accuracy  : 0.9304
    Macro F1  : 0.9303
    Precision : 0.9319
    Recall    : 0.9309

[2] MULTI-SEED STABILITY  (seeds: [42, 123, 2024])
    Mean F1 : 0.9220
    Std F1  : 0.0042
    Status  : STABLE

[3] ABLATION SUMMARY (8 variants)
    Best variant  : w/o RegGate               F1=0.9322
    Worst variant : MinimalBaseline           F1=0.9164
    Full ablation table → /content/hsenti_results/extended_ablation_table.csv

[4] OVERFITTING DIAGNOSIS
    Final train-val acc gap : +0.0716
    Interpretation          : Mild overfitting — consider increasing dropout or L2.

[5] ERROR ANALYSIS
    Overall accuracy        : 0.9304
    Negation text accuracy  : 0.8889
    Delta (neg vs overall)  : -0.0415
    Negation texts in test  : 162

In [ ]:
## ── XAI Cell: Explainable Attention Visualization ─────────────────────────

import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import arabic_reshaper
from bidi.algorithm import get_display

# ── CONFIG ────────────────────────────────────────────────────────────────
XAI_TEXT        = "ما شاء الله عليييه اكتشفته بالغلط أول مرة أشترك بقناة بعد فيديو واحد لازم أقيمهم فترة بس هذا أذهلني"   # ← edit freely
MAX_BARS        = 15
SENTIMENT_NAMES = {0: 'NEGATIVE', 1: 'POSITIVE'}
LABEL_COLOR     = {'NEGATIVE': '#e63946', 'POSITIVE': '#2a9d8f'}

# ── HELPERS ───────────────────────────────────────────────────────────────

def fix_arabic(text):
    return get_display(arabic_reshaper.reshape(str(text)))

def _encode(text):
    enc = tokenizer(
        text, max_length=MAX_LEN,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    return enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)

def _build_neg_mask(input_ids_list):
    mask = torch.zeros(MAX_LEN, dtype=torch.float)
    for i, tid in enumerate(input_ids_list[1:MAX_LEN-1], start=1):
        tok = tokenizer.decode([tid], skip_special_tokens=True).strip()
        if tok in NEGATION_MARKERS:
            mask[i] = 1.0
    return mask

def _merge_subwords(tokens_raw, scores_raw, attn_mask):
    words, wscores = [], []
    cur_word, cur_sc = '', []
    for tok, sc, valid in zip(tokens_raw, scores_raw, attn_mask):
        if valid == 0:
            break
        if tok in ('[CLS]', '[SEP]'):
            continue
        if tok.startswith('##'):
            cur_word += tok[2:]
            cur_sc.append(sc)
        else:
            if cur_word:
                words.append(fix_arabic(cur_word))
                wscores.append(float(np.mean(cur_sc)))
            cur_word = tok
            cur_sc   = [sc]
    if cur_word:
        words.append(fix_arabic(cur_word))
        wscores.append(float(np.mean(cur_sc)))
    return words, wscores

# ── INFERENCE: HIERARCHICAL MODEL ─────────────────────────────────────────

def xai_hierarchical(text):
    model.eval()
    ids, mask = _encode(text)
    neg = _build_neg_mask(ids[0].tolist()).unsqueeze(0).to(DEVICE)

    captured = {}
    def _hook(module, inp, out):
        hidden, attn_mask, neg_mask = inp[0], inp[1], inp[2]
        scores = module.attn(hidden).squeeze(-1)
        scores = scores + module.neg_weight * neg_mask
        scores = scores.masked_fill(attn_mask == 0, -1e9)
        captured['w'] = F.softmax(scores, dim=-1)[0].cpu().numpy()

    handle = model.neg_pooling.register_forward_hook(_hook)
    with torch.no_grad():
        sent_logits, _, _ = model(ids, mask, neg)
    handle.remove()

    probs = F.softmax(sent_logits, dim=-1)[0].cpu().numpy()
    pid   = int(probs.argmax())
    tokens_raw = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    words, wscores = _merge_subwords(tokens_raw, captured['w'], mask[0].cpu().numpy())
    return SENTIMENT_NAMES[pid], float(probs[pid]), words, wscores

# ── INFERENCE: BASELINE MODEL ─────────────────────────────────────────────

def xai_baseline(text):
    baseline_model.eval()
    ids, mask = _encode(text)

    original_attn = baseline_model.encoder.config._attn_implementation
    baseline_model.encoder.config._attn_implementation = 'eager'

    with torch.no_grad():
        enc_out = baseline_model.encoder(
            input_ids=ids, attention_mask=mask, output_attentions=True
        )
        cls       = enc_out.last_hidden_state[:, 0]
        logits    = baseline_model.classifier(cls)
        last_attn = enc_out.attentions[-1]
        cls_attn  = last_attn[0, :, 0, :].mean(0).cpu().numpy()

    baseline_model.encoder.config._attn_implementation = original_attn

    probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    pid   = int(probs.argmax())
    tokens_raw = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    words, wscores = _merge_subwords(tokens_raw, cls_attn, mask[0].cpu().numpy())
    return SENTIMENT_NAMES[pid], float(probs[pid]), words, wscores

# ── CHART ─────────────────────────────────────────────────────────────────

def _bar_chart(ax, words, scores, label, conf, title, color):
    pairs = sorted(zip(scores, words), reverse=True)[:MAX_BARS]
    pairs = sorted(pairs)
    sc    = [p[0] for p in pairs]
    tok   = [p[1] for p in pairs]
    total = sum(sc) or 1
    sc_n  = [s / total for s in sc]

    bars = ax.barh(tok, sc_n, color=color, edgecolor='none', height=0.65)
    for bar, val in zip(bars, sc_n):
        ax.text(bar.get_width() + max(sc_n) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

    ax.set_title(
        f'{title}\nPrediction: {label}  |  Confidence: {conf*100:.1f}%',
        fontsize=11, fontweight='bold',
        color=LABEL_COLOR.get(label, '#333333'), pad=10
    )
    ax.set_xlabel('Attention Score (normalized)', fontsize=9)
    ax.set_xlim(0, max(sc_n) * 1.22)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='y', labelsize=10)

# ── RUN ───────────────────────────────────────────────────────────────────

text_clean = preprocess_arabic(XAI_TEXT)
print(f'[XAI] Input      : {XAI_TEXT}')
print(f'[XAI] Preprocessed: {text_clean}\n')

h_label, h_conf, h_words, h_scores = xai_hierarchical(text_clean)
b_label, b_conf, b_words, b_scores = xai_baseline(text_clean)

print(f'[XAI] Hierarchical → {h_label} ({h_conf*100:.1f}%)')
print(f'[XAI] Baseline     → {b_label} ({b_conf*100:.1f}%)')

n_rows = min(max(len(h_words), len(b_words)), MAX_BARS)
fig    = plt.figure(figsize=(16, max(5, n_rows * 0.55 + 3)))
gs     = gridspec.GridSpec(1, 2, wspace=0.45)

_bar_chart(fig.add_subplot(gs[0]),
           h_words, h_scores, h_label, h_conf,
           'Hierarchical Model (Hiros)\nNegation-Aware Attention Pooling',
           LABEL_COLOR.get(h_label, '#0077b6'))

_bar_chart(fig.add_subplot(gs[1]),
           b_words, b_scores, b_label, b_conf,
           'Baseline MARBERTv2\nLast-Layer CLS Attention (mean heads)',
           LABEL_COLOR.get(b_label, '#0077b6'))

fig.suptitle(f'XAI — Token Attention Weights\nInput: {fix_arabic(XAI_TEXT)}',
             fontsize=12, fontweight='bold', y=1.02)

plt.tight_layout()
out_path = f'{RESULT_DIR}/xai_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'\n[XAI] Saved → {out_path}')

[XAI] Input      : ما شاء الله عليييه اكتشفته بالغلط أول مرة أشترك بقناة بعد فيديو واحد لازم أقيمهم فترة بس هذا أذهلني
[XAI] Preprocessed: ما شاء الله عليييه اكتشفته بالغلط أول مرة أشترك بقناة بعد فيديو واحد لازم أقيمهم فترة بس هذا أذهلني

[XAI] Hierarchical → POSITIVE (96.8%)
[XAI] Baseline     → POSITIVE (98.2%)

[XAI] Saved → /content/hsenti_results/xai_comparison.png
